# CNN classifier-based density-ratio estimation

This notebook preserves the original patch-size and convolution experiment
variants. Each large experiment cell is self-contained; run only the section
you intend to reproduce. Generated outputs have been removed for a clean Git
history.

Run the configuration cell first. Override its paths with the
`DRE_PROJECT_ROOT`, `DRE_DATA_ROOT`, `DRE_OUTPUT_ROOT`, `DRE_RASTER_DIR`, or
`DRE_LANDMASK_PATH` environment variables when needed.


In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = Path(os.environ.get("DRE_PROJECT_ROOT", Path.cwd())).expanduser().resolve()
DATA_ROOT = Path(os.environ.get("DRE_DATA_ROOT", PROJECT_ROOT / "data" / "processed")).expanduser().resolve()
OUTPUT_ROOT = Path(os.environ.get("DRE_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "dre_approaches")).expanduser().resolve()
RASTER_DIR = Path(os.environ.get("DRE_RASTER_DIR", DATA_ROOT / "covariate_rasters")).expanduser().resolve()
LANDMASK_PATH = Path(os.environ.get("DRE_LANDMASK_PATH", DATA_ROOT / "landmask.tif")).expanduser().resolve()

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print("Project root:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_3_ens_by_logratio_pconv_stand")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask: (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    # Apply noise ONLY on valid pixels
    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        m = (mask > 0).to(dtype=values.dtype)
        values = values + noise * m

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# NEW: Channel-aware input partial conv + spatial partial conv
# =========================================================
class ChannelwisePartialConv2d(nn.Module):
    """
    Channel-aware partial conv for the *input covariate* layer.
    Fixes the "any-channel-valid" mask collapse by using (B,C,H,W) masks.

    For each input channel c:
      m_sum_c = sum(mask_c under kernel)
      x_norm_c = x_c * mask_c * (kernel_area / (m_sum_c + eps))
      x_norm_c = 0 where m_sum_c == 0

    After that, a standard Conv2d mixes channels.
    Returns y and a spatial mask m_spatial (B,1,H,W) indicating any covariate support.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)

        w = torch.ones(in_channels, 1, kH, kW)
        self.register_buffer("mask_weight", w)

        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x,m: (B,C,H,W)
        if m is None:
            m = torch.ones_like(x)
        if x.dim() != 4 or m.dim() != 4:
            raise ValueError(f"Expected x,m 4D. Got x={x.shape}, m={m.shape}")
        if m.shape != x.shape:
            raise ValueError(f"Channelwise layer expects mask same shape as x. Got x={x.shape}, m={m.shape}")

        m = (m > 0).to(dtype=x.dtype)

        # (B,C,H,W): per-channel valid counts
        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))

        scale = self.kernel_area / (m_sum + self.eps)
        x_norm = x * m * scale
        x_norm = x_norm * (m_sum > 0).to(dtype=x.dtype)

        y = self.conv(x_norm)

        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(dtype=x.dtype)  # (B,1,H,W)

        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    """
    Standard partial conv in feature space with a 1-channel spatial mask (B,1,H,W).
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x: (B,C,H,W), m: (B,1,H,W)
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        if m.dim() != 4 or m.size(1) != 1:
            raise ValueError(f"Spatial mask must be (B,1,H,W). Got {m.shape}")

        m = (m > 0).to(dtype=x.dtype)
        x_masked = x * m
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride, padding=self.padding)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    """First block: channel-aware partial conv (input covariates)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)  # m is spatial (B,1,H,W)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class CPConvBlock(nn.Module):
    """Later blocks: spatial partial conv in feature space."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = SpatialPartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.p(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


# =========================================================
# UPDATED ENCODER: channel-aware at input, patch-size agnostic
# =========================================================
class PatchEncoderCPConv5(nn.Module):
    """
    Patch-size agnostic encoder for multi-S experiments:
      - No pooling
      - AdaptiveAvgPool2d(1) at end
      - Fixes channel-mask collapse at input via ChannelwisePartialConv2d
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.b3 = CPConvBlock(16, 32)
        self.b4 = CPConvBlock(32, 32)
        self.b5 = CPConvBlock(32, 32)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,S,S), x_mask: (B,C,S,S)
        m_cov = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m_cov)  # m becomes spatial (B,1,S,S)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x)          # (B,32,1,1)
        z = self.proj(h)         # (B,emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoderCPConv5(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_stand_patch_new")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask: (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    # Apply noise ONLY on valid pixels
    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        m = (mask > 0).to(dtype=values.dtype)
        values = values + noise * m

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# NEW: Channel-aware input partial conv + spatial partial conv
# =========================================================
class ChannelwisePartialConv2d(nn.Module):
    """
    Channel-aware partial conv for the *input covariate* layer.
    Fixes the "any-channel-valid" mask collapse by using (B,C,H,W) masks.

    For each input channel c:
      m_sum_c = sum(mask_c under kernel)
      x_norm_c = x_c * mask_c * (kernel_area / (m_sum_c + eps))
      x_norm_c = 0 where m_sum_c == 0

    After that, a standard Conv2d mixes channels.
    Returns y and a spatial mask m_spatial (B,1,H,W) indicating any covariate support.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)

        w = torch.ones(in_channels, 1, kH, kW)
        self.register_buffer("mask_weight", w)

        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x,m: (B,C,H,W)
        if m is None:
            m = torch.ones_like(x)
        if x.dim() != 4 or m.dim() != 4:
            raise ValueError(f"Expected x,m 4D. Got x={x.shape}, m={m.shape}")
        if m.shape != x.shape:
            raise ValueError(f"Channelwise layer expects mask same shape as x. Got x={x.shape}, m={m.shape}")

        m = (m > 0).to(dtype=x.dtype)

        # (B,C,H,W): per-channel valid counts
        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))

        scale = self.kernel_area / (m_sum + self.eps)
        x_norm = x * m * scale
        x_norm = x_norm * (m_sum > 0).to(dtype=x.dtype)

        y = self.conv(x_norm)

        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(dtype=x.dtype)  # (B,1,H,W)

        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    """
    Standard partial conv in feature space with a 1-channel spatial mask (B,1,H,W).
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x: (B,C,H,W), m: (B,1,H,W)
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        if m.dim() != 4 or m.size(1) != 1:
            raise ValueError(f"Spatial mask must be (B,1,H,W). Got {m.shape}")

        m = (m > 0).to(dtype=x.dtype)
        x_masked = x * m
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride, padding=self.padding)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    """First block: channel-aware partial conv (input covariates)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)  # m is spatial (B,1,H,W)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class CPConvBlock(nn.Module):
    """Later blocks: spatial partial conv in feature space."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = SpatialPartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.p(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


# =========================================================
# UPDATED ENCODER: channel-aware at input, patch-size agnostic
# =========================================================
class PatchEncoderCPConv5(nn.Module):
    """
    Patch-size agnostic encoder for multi-S experiments:
      - No pooling
      - AdaptiveAvgPool2d(1) at end
      - Fixes channel-mask collapse at input via ChannelwisePartialConv2d
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.b3 = CPConvBlock(16, 32)
        self.b4 = CPConvBlock(32, 32)
        self.b5 = CPConvBlock(32, 32)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,S,S), x_mask: (B,C,S,S)
        m_cov = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m_cov)  # m becomes spatial (B,1,S,S)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x)          # (B,32,1,1)
        z = self.proj(h)         # (B,emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoderCPConv5(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_New_13")
TEST_DIR = str(DATA_ROOT / "test_patches_New_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_stand_patch_new_try")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask: (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    # Apply noise ONLY on valid pixels
    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        m = (mask > 0).to(dtype=values.dtype)
        values = values + noise * m

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# NEW: Channel-aware input partial conv + spatial partial conv
# =========================================================
class ChannelwisePartialConv2d(nn.Module):
    """
    Channel-aware partial conv for the *input covariate* layer.
    Fixes the "any-channel-valid" mask collapse by using (B,C,H,W) masks.

    For each input channel c:
      m_sum_c = sum(mask_c under kernel)
      x_norm_c = x_c * mask_c * (kernel_area / (m_sum_c + eps))
      x_norm_c = 0 where m_sum_c == 0

    After that, a standard Conv2d mixes channels.
    Returns y and a spatial mask m_spatial (B,1,H,W) indicating any covariate support.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)

        w = torch.ones(in_channels, 1, kH, kW)
        self.register_buffer("mask_weight", w)

        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x,m: (B,C,H,W)
        if m is None:
            m = torch.ones_like(x)
        if x.dim() != 4 or m.dim() != 4:
            raise ValueError(f"Expected x,m 4D. Got x={x.shape}, m={m.shape}")
        if m.shape != x.shape:
            raise ValueError(f"Channelwise layer expects mask same shape as x. Got x={x.shape}, m={m.shape}")

        m = (m > 0).to(dtype=x.dtype)

        # (B,C,H,W): per-channel valid counts
        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))

        scale = self.kernel_area / (m_sum + self.eps)
        x_norm = x * m * scale
        x_norm = x_norm * (m_sum > 0).to(dtype=x.dtype)

        y = self.conv(x_norm)

        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(dtype=x.dtype)  # (B,1,H,W)

        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    """
    Standard partial conv in feature space with a 1-channel spatial mask (B,1,H,W).
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x: (B,C,H,W), m: (B,1,H,W)
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        if m.dim() != 4 or m.size(1) != 1:
            raise ValueError(f"Spatial mask must be (B,1,H,W). Got {m.shape}")

        m = (m > 0).to(dtype=x.dtype)
        x_masked = x * m
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride, padding=self.padding)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    """First block: channel-aware partial conv (input covariates)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)  # m is spatial (B,1,H,W)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class CPConvBlock(nn.Module):
    """Later blocks: spatial partial conv in feature space."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = SpatialPartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.p(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


# =========================================================
# UPDATED ENCODER: channel-aware at input, patch-size agnostic
# =========================================================
class PatchEncoderCPConv5(nn.Module):
    """
    Patch-size agnostic encoder for multi-S experiments:
      - No pooling
      - AdaptiveAvgPool2d(1) at end
      - Fixes channel-mask collapse at input via ChannelwisePartialConv2d
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.b3 = CPConvBlock(16, 32)
        self.b4 = CPConvBlock(32, 32)
        self.b5 = CPConvBlock(32, 32)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,S,S), x_mask: (B,C,S,S)
        m_cov = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m_cov)  # m becomes spatial (B,1,S,S)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x)          # (B,32,1,1)
        z = self.proj(h)         # (B,emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoderCPConv5(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_2j")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (Liu et al., 2018).

    Standard mode (channelwise_mask=False):
        Mask is a single spatial channel (B, 1, H, W).
        y = conv(x * m) * (kH*kW / sum_m)
        m_out = 1 where sum_m > 0

    Channel-wise mode (channelwise_mask=True, first layer only):
        Mask is per-channel (B, C_in, H, W) so that missingness
        can differ across covariate channels.  Each input value
        is zeroed independently per channel.  Rescaling uses the
        count of valid entries across ALL input channels within
        the kernel window (kernel_area = C_in * kH * kW).

        The output mask remains per-channel (B, C_in, H_out, W_out)
        so that downstream code can decide when to collapse it.
        The encoder collapses it to a single spatial mask after
        the first block.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask
        self.in_channels = in_channels

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            # Kernel for counting valid entries across all C_in channels
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
            # Per-channel kernels for propagating each channel's mask
            # Shape: (C_in, 1, kH, kW) — depthwise conv to pool each channel
            self.register_buffer("mask_kernel_perchan",
                                 torch.ones(in_channels, 1, kH, kW))
        else:
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if self.channelwise_mask:
            # ---- Channel-wise partial convolution ----
            # m: (B, C_in, H, W) with per-channel validity
            m_float = (m > 0).to(dtype=x.dtype)

            # Zero out invalid entries per channel
            x_masked = x * m_float

            # Standard Conv2d on masked input
            y = self.conv(x_masked)  # (B, C_out, H_out, W_out)

            with torch.no_grad():
                # Total valid count across all channels for rescaling
                # mask_kernel: (1, C_in, kH, kW) -> (B, 1, H_out, W_out)
                m_sum_total = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )  # (B, 1, H_out, W_out)

                # Per-channel mask propagation (depthwise conv)
                # mask_kernel_perchan: (C_in, 1, kH, kW) -> groups=C_in
                # Output: (B, C_in, H_out, W_out)
                m_out_perchan = F.conv2d(
                    m_float, self.mask_kernel_perchan,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation, groups=self.in_channels
                )
                m_out_perchan = (m_out_perchan > 0).to(dtype=x.dtype)

            # Rescale using total valid count
            valid_anywhere = (m_sum_total > 0).to(dtype=x.dtype)
            scale = self.kernel_area / (m_sum_total + self.eps)
            y = y * scale * valid_anywhere

            # Return per-channel output mask (B, C_in, H_out, W_out)
            return y, m_out_perchan

        else:
            # ---- Standard spatial partial convolution ----
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


# =========================================================
# ENCODER — U-Net-style bottleneck built from the same
# mask-aware partial-convolution blocks described in the
# paper.  All building blocks are unchanged:
#   - channel-wise mask in the first layer, per-channel
#     propagation, explicit collapse after b1
#   - 3x3 partial-conv blocks + BN + ReLU
#   - mask-aware pooling and global average pooling
#   - one dilated partial convolution (dilation=2)
#
# What changes is the encoder depth and the progressive
# contraction (like the encoder half of a U-Net):
#
#   Stage 1 (13x13):  b1  C_in -> 16  channel-wise mask
#                     collapse mask -> (B,1,H,W)
#                     b2  16 -> 16
#   Pool1:            13 -> 6
#   Stage 2 (6x6):    b3  16 -> 32
#                     b4  32 -> 32   dilated (dilation=2)
#   Pool2:            6  -> 3
#   Bottleneck (3x3): b5  32 -> 64
#   GAP:              mask-aware global average pool
#   proj:             Linear(64, emb_dim) + ReLU
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()

        # ---------- Stage 1: 13x13 ----------
        # First block: channel-wise mask (per-covariate missingness)
        self.b1 = MaskedConvBlock(
            in_value_channels, 16, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        # Second block: standard spatial mask
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)

        # ---------- Pool: 13 -> 6 ----------
        self.pool1 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 2: 6x6 ----------
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        # Dilated partial convolution
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=2, dilation=2)

        # ---------- Pool: 6 -> 3 ----------
        self.pool2 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 3 / Bottleneck: 3x3 ----------
        self.b5 = MaskedConvBlock(32, 64, k=3, s=1, p=1)

        # ---------- Aggregation ----------
        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(64, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val:  (B, C, 13, 13)
        # x_mask: (B, C, 13, 13) — per-channel validity mask
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # ---- Stage 1 (13x13) ----
        # b1: channel-wise partial conv, mask stays (B, C_in, H, W)
        x, m = self.b1(x_val, m)

        # Collapse mask to single spatial channel
        # m: (B, C_in, H, W) -> (B, 1, H, W)
        m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)

        # b2: spatial partial conv
        x, m = self.b2(x, m)

        # ---- Pool: 13 -> 6 ----
        x, m = self.pool1(x, m)

        # ---- Stage 2 (6x6) ----
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)  # dilated

        # ---- Pool: 6 -> 3 ----
        x, m = self.pool2(x, m)

        # ---- Bottleneck (3x3) ----
        x, m = self.b5(x, m)

        # ---- Aggregation ----
        h = self.gap(x, m)  # (B, 64)
        z = self.proj(h)     # (B, emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)
    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")

In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_3j")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True

# Ranking loss (combined objective: L = L_BCE + LAMBDA_RANK * L_rank)
LAMBDA_RANK = 1.0
RANK_TEMPERATURE = 1.0


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (Liu et al., 2018).

    Standard mode (channelwise_mask=False):
        Mask is a single spatial channel (B, 1, H, W).
        y = conv(x * m) * (kH*kW / sum_m)
        m_out = 1 where sum_m > 0

    Channel-wise mode (channelwise_mask=True, first layer only):
        Mask is per-channel (B, C_in, H, W) so that missingness
        can differ across covariate channels.  Each input value
        is zeroed independently per channel.  Rescaling uses the
        count of valid entries across ALL input channels within
        the kernel window (kernel_area = C_in * kH * kW).

        The output mask remains per-channel (B, C_in, H_out, W_out)
        so that downstream code can decide when to collapse it.
        The encoder collapses it to a single spatial mask after
        the first block.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask
        self.in_channels = in_channels

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            # Kernel for counting valid entries across all C_in channels
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
            # Per-channel kernels for propagating each channel's mask
            # Shape: (C_in, 1, kH, kW) — depthwise conv to pool each channel
            self.register_buffer("mask_kernel_perchan",
                                 torch.ones(in_channels, 1, kH, kW))
        else:
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if self.channelwise_mask:
            # ---- Channel-wise partial convolution ----
            # m: (B, C_in, H, W) with per-channel validity
            m_float = (m > 0).to(dtype=x.dtype)

            # Zero out invalid entries per channel
            x_masked = x * m_float

            # Standard Conv2d on masked input
            y = self.conv(x_masked)  # (B, C_out, H_out, W_out)

            with torch.no_grad():
                # Total valid count across all channels for rescaling
                # mask_kernel: (1, C_in, kH, kW) -> (B, 1, H_out, W_out)
                m_sum_total = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )  # (B, 1, H_out, W_out)

                # Per-channel mask propagation (depthwise conv)
                # mask_kernel_perchan: (C_in, 1, kH, kW) -> groups=C_in
                # Output: (B, C_in, H_out, W_out)
                m_out_perchan = F.conv2d(
                    m_float, self.mask_kernel_perchan,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation, groups=self.in_channels
                )
                m_out_perchan = (m_out_perchan > 0).to(dtype=x.dtype)

            # Rescale using total valid count
            valid_anywhere = (m_sum_total > 0).to(dtype=x.dtype)
            scale = self.kernel_area / (m_sum_total + self.eps)
            y = y * scale * valid_anywhere

            # Return per-channel output mask (B, C_in, H_out, W_out)
            return y, m_out_perchan

        else:
            # ---- Standard spatial partial convolution ----
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


# =========================================================
# ENCODER — U-Net-style bottleneck built from the same
# mask-aware partial-convolution blocks described in the
# paper.  All building blocks are unchanged:
#   - channel-wise mask in the first layer, per-channel
#     propagation, explicit collapse after b1
#   - 3x3 partial-conv blocks + BN + ReLU
#   - mask-aware pooling and global average pooling
#   - one dilated partial convolution (dilation=2)
#
# What changes is the encoder depth and the progressive
# contraction (like the encoder half of a U-Net):
#
#   Stage 1 (13x13):  b1  C_in -> 16  channel-wise mask
#                     collapse mask -> (B,1,H,W)
#                     b2  16 -> 16
#   Pool1:            13 -> 6
#   Stage 2 (6x6):    b3  16 -> 32
#                     b4  32 -> 32   dilated (dilation=2)
#   Pool2:            6  -> 3
#   Bottleneck (3x3): b5  32 -> 64
#   GAP:              mask-aware global average pool
#   proj:             Linear(64, emb_dim) + ReLU
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()

        # ---------- Stage 1: 13x13 ----------
        # First block: channel-wise mask (per-covariate missingness)
        self.b1 = MaskedConvBlock(
            in_value_channels, 16, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        # Second block: standard spatial mask
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)

        # ---------- Pool: 13 -> 6 ----------
        self.pool1 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 2: 6x6 ----------
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        # Dilated partial convolution
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=2, dilation=2)

        # ---------- Pool: 6 -> 3 ----------
        self.pool2 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 3 / Bottleneck: 3x3 ----------
        self.b5 = MaskedConvBlock(32, 64, k=3, s=1, p=1)

        # ---------- Aggregation ----------
        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(64, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val:  (B, C, 13, 13)
        # x_mask: (B, C, 13, 13) — per-channel validity mask
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # ---- Stage 1 (13x13) ----
        # b1: channel-wise partial conv, mask stays (B, C_in, H, W)
        x, m = self.b1(x_val, m)

        # Collapse mask to single spatial channel
        # m: (B, C_in, H, W) -> (B, 1, H, W)
        m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)

        # b2: spatial partial conv
        x, m = self.b2(x, m)

        # ---- Pool: 13 -> 6 ----
        x, m = self.pool1(x, m)

        # ---- Stage 2 (6x6) ----
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)  # dilated

        # ---- Pool: 6 -> 3 ----
        x, m = self.pool2(x, m)

        # ---- Bottleneck (3x3) ----
        x, m = self.b5(x, m)

        # ---- Aggregation ----
        h = self.gap(x, m)  # (B, 64)
        z = self.proj(h)     # (B, emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Pairwise ranking loss
# =========================================================
def pairwise_ranking_loss(logits: torch.Tensor, y: torch.Tensor,
                          temperature: float = 1.0) -> torch.Tensor:
    """
    Pairwise ranking loss that penalises cases where a background
    sample scores above a presence sample.

    For each presence-background pair (i, j) in the mini-batch:
        L_rank = (1/|P|) * sum log(1 + exp(-(s_i - s_j) / tau))

    where s_i is a presence logit, s_j is a background logit,
    tau is a temperature controlling sensitivity, and P is the
    set of all presence-background pairs.
    """
    pres_mask = (y == 1)
    bg_mask = (y == 0)

    s_pres = logits[pres_mask]
    s_bg = logits[bg_mask]

    if s_pres.numel() == 0 or s_bg.numel() == 0:
        return torch.tensor(0.0, device=logits.device, dtype=logits.dtype)

    # All pairwise differences: (n_pres, n_bg)
    diff = s_pres.unsqueeze(1) - s_bg.unsqueeze(0)

    loss = torch.log1p(torch.exp(-diff / temperature))
    return loss.mean()


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss_bce = bce(logits, yb)
                if train and LAMBDA_RANK > 0:
                    loss_rank = pairwise_ranking_loss(logits, yb, temperature=RANK_TEMPERATURE)
                    loss = loss_bce + LAMBDA_RANK * loss_rank
                else:
                    loss = loss_bce
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)
    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")

In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_New_13")
TEST_DIR = str(DATA_ROOT / "test_patches_New_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_4j")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 5e-3   # increased from 1e-3 for stronger L2 regularization

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.5          # increased from 0.35
SPATIAL_DROPOUT = 0.15 # drop entire feature maps in the encoder

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08
MIXUP_ALPHA = 0.2  # Beta distribution parameter for mixup (0 = disabled)

# Label smoothing: softens 0/1 targets to prevent overconfidence
LABEL_SMOOTH = 0.05  # y=1 becomes 0.95, y=0 becomes 0.05

# Ranking loss (combined objective: L = L_BCE + LAMBDA_RANK * L_rank)
LAMBDA_RANK = 1.0
RANK_TEMPERATURE = 1.0

# Selection rule
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.08  # wider than 0.03 to not exclude high-Boyce epochs

# Boyce smoothing for model selection
BOYCE_EMA_ALPHA = 0.4

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble
ENSEMBLE_IN_LOGSPACE = True
ENSEMBLE_WEIGHT_BY = "boyce"  # "boyce" or "loss"


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    # Random 90-degree rotation (0, 90, 180, 270)
    k = torch.randint(0, 4, (1,)).item()
    if k > 0:
        values = torch.rot90(values, k, dims=[1, 2])
        mask = torch.rot90(mask, k, dims=[1, 2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution (Liu et al., 2018).

    Standard mode (channelwise_mask=False):
        Mask is a single spatial channel (B, 1, H, W).
        y = conv(x * m) * (kH*kW / sum_m)
        m_out = 1 where sum_m > 0

    Channel-wise mode (channelwise_mask=True, first layer only):
        Mask is per-channel (B, C_in, H, W) so that missingness
        can differ across covariate channels.  Each input value
        is zeroed independently per channel.  Rescaling uses the
        count of valid entries across ALL input channels within
        the kernel window (kernel_area = C_in * kH * kW).

        The output mask remains per-channel (B, C_in, H_out, W_out)
        so that downstream code can decide when to collapse it.
        The encoder collapses it to a single spatial mask after
        the first block.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask
        self.in_channels = in_channels

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            # Kernel for counting valid entries across all C_in channels
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
            # Per-channel kernels for propagating each channel's mask
            # Shape: (C_in, 1, kH, kW) — depthwise conv to pool each channel
            self.register_buffer("mask_kernel_perchan",
                                 torch.ones(in_channels, 1, kH, kW))
        else:
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if self.channelwise_mask:
            # ---- Channel-wise partial convolution ----
            # m: (B, C_in, H, W) with per-channel validity
            m_float = (m > 0).to(dtype=x.dtype)

            # Zero out invalid entries per channel
            x_masked = x * m_float

            # Standard Conv2d on masked input
            y = self.conv(x_masked)  # (B, C_out, H_out, W_out)

            with torch.no_grad():
                # Total valid count across all channels for rescaling
                # mask_kernel: (1, C_in, kH, kW) -> (B, 1, H_out, W_out)
                m_sum_total = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )  # (B, 1, H_out, W_out)

                # Per-channel mask propagation (depthwise conv)
                # mask_kernel_perchan: (C_in, 1, kH, kW) -> groups=C_in
                # Output: (B, C_in, H_out, W_out)
                m_out_perchan = F.conv2d(
                    m_float, self.mask_kernel_perchan,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation, groups=self.in_channels
                )
                m_out_perchan = (m_out_perchan > 0).to(dtype=x.dtype)

            # Rescale using total valid count
            valid_anywhere = (m_sum_total > 0).to(dtype=x.dtype)
            scale = self.kernel_area / (m_sum_total + self.eps)
            y = y * scale * valid_anywhere

            # Return per-channel output mask (B, C_in, H_out, W_out)
            return y, m_out_perchan

        else:
            # ---- Standard spatial partial convolution ----
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


class SpatialDropout2d(nn.Module):
    """
    Drops entire feature maps (channels) rather than individual
    neurons.  More appropriate than element-wise dropout for CNN
    feature maps where adjacent activations are highly correlated.
    Forces the encoder to distribute information across channels
    instead of relying on a few dominant ones.
    """
    def __init__(self, p: float = 0.1):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0:
            return x
        # x: (B, C, H, W) — drop entire (H, W) slices
        mask = torch.ones(x.size(0), x.size(1), 1, 1, device=x.device, dtype=x.dtype)
        mask = F.dropout(mask, p=self.p, training=True)
        return x * mask


# =========================================================
# ENCODER — paper-faithful architecture with added spatial
# dropout for regularization.
#
# Architecture unchanged from the paper:
#   b1: partial conv (C_in -> 16) with channel-wise mask
#       -> mask stays (B, C_in, H, W) through the conv
#       -> collapsed to (B, 1, H, W) after b1
#   b2: partial conv (16 -> 16) with spatial mask
#   pool: mask-aware max pool 2x2 (13 -> 6)
#   b3: dilated partial conv (16 -> 32) dilation=2
#   GAP: mask-aware global average pool
#   proj: Linear(32, emb_dim) + ReLU
#
# Added for generalization:
#   - SpatialDropout2d between conv blocks (drops entire
#     feature maps, forcing the encoder to spread information
#     across channels rather than relying on a few dominant
#     covariate-derived features)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32,
                 spatial_dropout: float = 0.15):
        super().__init__()
        # First block: channel-wise mask (per-covariate missingness)
        self.b1 = MaskedConvBlock(
            in_value_channels, 16, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        self.sdrop1 = SpatialDropout2d(spatial_dropout)

        # Second block: standard spatial mask
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.sdrop2 = SpatialDropout2d(spatial_dropout)

        # Mask-aware pooling: 13 -> 6
        self.pool = MaskedMaxPool2d(2, 2)

        # Dilated partial convolution block
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=2, dilation=2)
        self.sdrop3 = SpatialDropout2d(spatial_dropout)

        # Mask-aware global average pooling
        self.gap = MaskedGlobalAvgPool()

        # Linear projection to embedding
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val:  (B, C, 13, 13)
        # x_mask: (B, C, 13, 13) — per-channel validity mask
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # Block 1: channel-wise partial conv
        x, m = self.b1(x_val, m)
        x = self.sdrop1(x)

        # Collapse mask to single spatial channel
        m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)

        # Block 2: standard spatial partial conv
        x, m = self.b2(x, m)
        x = self.sdrop2(x)

        # Pooling: 13 -> 6
        x, m = self.pool(x, m)

        # Block 3: dilated partial conv (dilation=2)
        x, m = self.b3(x, m)
        x = self.sdrop3(x)

        # Mask-aware global average pool
        h = self.gap(x, m)  # (B, 32)

        # Linear projection
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64),
                 dropout=0.2, clip_logits=10.0, spatial_dropout=0.15):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim,
                                      spatial_dropout=spatial_dropout)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Pairwise ranking loss
# =========================================================
def pairwise_ranking_loss(logits: torch.Tensor, y: torch.Tensor,
                          temperature: float = 1.0) -> torch.Tensor:
    """
    Penalises cases where a background sample scores above a
    presence sample: L = mean(log(1 + exp(-(s_pres - s_bg)/tau)))
    """
    pres_mask = (y > 0.5)  # works with label-smoothed targets
    bg_mask = ~pres_mask

    s_pres = logits[pres_mask]
    s_bg = logits[bg_mask]

    if s_pres.numel() == 0 or s_bg.numel() == 0:
        return torch.tensor(0.0, device=logits.device, dtype=logits.dtype)

    diff = s_pres.unsqueeze(1) - s_bg.unsqueeze(0)
    return torch.log1p(torch.exp(-diff / temperature)).mean()


# =========================================================
# Mixup for patches
# =========================================================
def mixup_batch(xv, xm, yb, alpha=0.2):
    """
    Mixup augmentation: interpolates between random pairs
    within the batch.  Creates synthetic environmental
    gradients that force the model to learn smooth
    suitability transitions.
    """
    if alpha <= 0:
        return xv, xm, yb

    lam = np.random.beta(alpha, alpha)
    lam = max(lam, 1.0 - lam)  # ensure lam >= 0.5

    idx = torch.randperm(xv.size(0), device=xv.device)

    xv_mix = lam * xv + (1.0 - lam) * xv[idx]
    xm_mix = torch.maximum(xm, xm[idx])  # union of valid pixels
    yb_mix = lam * yb + (1.0 - lam) * yb[idx]

    return xv_mix, xm_mix, yb_mix


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
        spatial_dropout=SPATIAL_DROPOUT,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)

            # ---- Training-only transforms ----
            if train:
                # Label smoothing
                if LABEL_SMOOTH > 0:
                    yb = yb * (1.0 - LABEL_SMOOTH) + 0.5 * LABEL_SMOOTH

                # Mixup
                if MIXUP_ALPHA > 0:
                    xv, xm, yb = mixup_batch(xv, xm, yb, alpha=MIXUP_ALPHA)

                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss_bce = bce(logits, yb)
                if train and LAMBDA_RANK > 0:
                    loss_rank = pairwise_ranking_loss(logits, yb,
                                                     temperature=RANK_TEMPERATURE)
                    loss = loss_bce + LAMBDA_RANK * loss_rank
                else:
                    loss = loss_bce
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0
    boyce_ema = None

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        # Smoothed Boyce for selection
        raw_boyce = float(va["Boyce"]) if np.isfinite(va["Boyce"]) else 0.0
        if boyce_ema is None:
            boyce_ema = raw_boyce
        else:
            boyce_ema = BOYCE_EMA_ALPHA * raw_boyce + (1.0 - BOYCE_EMA_ALPHA) * boyce_ema

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f} (ema {boyce_ema:.4f})"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "boyce_ema": float(boyce_ema),
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    # Select: within loss window, pick best smoothed Boyce
    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce_ema"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f} (ema={best['boyce_ema']:.4f}), AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "Boyce_ema": best["boyce_ema"],
                "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
            spatial_dropout=SPATIAL_DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)
    fold_aucs = []
    fold_losses = []
    fold_boyces = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        fold_boyces.append(best_val.get("Boyce_ema", best_val["Boyce"]))
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)
    print("[CV] per-fold chosen Boyce (ema):", fold_boyces)

    loss_arr = np.asarray(fold_losses, dtype=float)
    boyce_arr = np.asarray(fold_boyces, dtype=float)

    if ENSEMBLE_WEIGHT_BY == "boyce":
        if np.isfinite(boyce_arr).all() and boyce_arr.size > 0:
            b_shifted = boyce_arr - boyce_arr.min() + 0.01
            w = b_shifted / b_shifted.sum()
            weights = w
            print(f"\n[Ensemble] Boyce-weighted: {weights}")
        else:
            weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
            print("\n[Ensemble] Boyce values invalid; using equal weights.")
    else:
        if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
            weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
            print("\n[Ensemble] Losses invalid; using equal weights.")
        else:
            loss_min = float(np.min(loss_arr))
            w = np.exp(-(loss_arr - loss_min))
            w = w / w.sum()
            weights = w
            print(f"\n[Ensemble] Loss-based weights: {weights}")

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_boyces.npy"), boyce_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_boyces.npy, fold_weights_loss.npy, oof_score01.npy")

    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")

In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_4j")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_13_pconv_4j.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 13
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.5
SPATIAL_DROPOUT = 0.15

WRITE_LOGR = False
ENSEMBLE_IN_LOGSPACE = True

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — paper-faithful architecture
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution.

    Channel-wise mode (channelwise_mask=True, first layer only):
        Mask is per-channel (B, C_in, H, W).  Each input value is
        zeroed independently per channel.  Rescaling uses the count
        of valid entries across ALL input channels within the kernel
        window (kernel_area = C_in * kH * kW).

        The output mask stays per-channel (B, C_in, H_out, W_out);
        the encoder collapses it to a single spatial mask after the
        first block.

    Standard mode (channelwise_mask=False):
        Mask is a single spatial channel (B, 1, H, W).
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask
        self.in_channels = in_channels

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
            self.register_buffer("mask_kernel_perchan",
                                 torch.ones(in_channels, 1, kH, kW))
        else:
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if self.channelwise_mask:
            m_float = (m > 0).to(dtype=x.dtype)
            x_masked = x * m_float
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum_total = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )

                m_out_perchan = F.conv2d(
                    m_float, self.mask_kernel_perchan,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation, groups=self.in_channels
                )
                m_out_perchan = (m_out_perchan > 0).to(dtype=x.dtype)

            valid_anywhere = (m_sum_total > 0).to(dtype=x.dtype)
            scale = self.kernel_area / (m_sum_total + self.eps)
            y = y * scale * valid_anywhere

            return y, m_out_perchan

        else:
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class SpatialDropout2d(nn.Module):
    def __init__(self, p: float = 0.1):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not self.training or self.p == 0:
            return x
        mask = torch.ones(x.size(0), x.size(1), 1, 1, device=x.device, dtype=x.dtype)
        mask = F.dropout(mask, p=self.p, training=True)
        return x * mask


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32,
                 spatial_dropout: float = 0.15):
        super().__init__()
        self.b1 = MaskedConvBlock(
            in_value_channels, 16, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        self.sdrop1 = SpatialDropout2d(spatial_dropout)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.sdrop2 = SpatialDropout2d(spatial_dropout)
        self.pool = MaskedMaxPool2d(2, 2)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=2, dilation=2)
        self.sdrop3 = SpatialDropout2d(spatial_dropout)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x = self.sdrop1(x)
        m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)

        x, m = self.b2(x, m)
        x = self.sdrop2(x)

        x, m = self.pool(x, m)

        x, m = self.b3(x, m)
        x = self.sdrop3(x)

        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
        spatial_dropout: float = 0.15,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim,
                                      spatial_dropout=spatial_dropout)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
            spatial_dropout=SPATIAL_DROPOUT,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels})."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE ({PATCH_SIZE})."
        )

    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )

In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_4j")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_13_pconv_4j.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 13
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

WRITE_LOGR = False
ENSEMBLE_IN_LOGSPACE = True

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — paper-faithful architecture
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution.

    Channel-wise mode (channelwise_mask=True, first layer only):
        Mask is per-channel (B, C_in, H, W).  Each input value is
        zeroed independently per channel.  Rescaling uses the count
        of valid entries across ALL input channels within the kernel
        window (kernel_area = C_in * kH * kW).

        The output mask stays per-channel (B, C_in, H_out, W_out);
        the encoder collapses it to a single spatial mask after the
        first block.

    Standard mode (channelwise_mask=False):
        Mask is a single spatial channel (B, 1, H, W).
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask
        self.in_channels = in_channels

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
            self.register_buffer("mask_kernel_perchan",
                                 torch.ones(in_channels, 1, kH, kW))
        else:
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D. Got {m.shape}")

        if self.channelwise_mask:
            m_float = (m > 0).to(dtype=x.dtype)
            x_masked = x * m_float
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum_total = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )

                m_out_perchan = F.conv2d(
                    m_float, self.mask_kernel_perchan,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation, groups=self.in_channels
                )
                m_out_perchan = (m_out_perchan > 0).to(dtype=x.dtype)

            valid_anywhere = (m_sum_total > 0).to(dtype=x.dtype)
            scale = self.kernel_area / (m_sum_total + self.eps)
            y = y * scale * valid_anywhere

            return y, m_out_perchan

        else:
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding,
                    dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)
        return num / den


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()

        # Stage 1: 13x13
        self.b1 = MaskedConvBlock(
            in_value_channels, 16, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)

        # Pool: 13 -> 6
        self.pool1 = MaskedMaxPool2d(2, 2)

        # Stage 2: 6x6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=2, dilation=2)

        # Pool: 6 -> 3
        self.pool2 = MaskedMaxPool2d(2, 2)

        # Bottleneck: 3x3
        self.b5 = MaskedConvBlock(32, 64, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(64, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # Stage 1 (13x13)
        x, m = self.b1(x_val, m)
        m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        x, m = self.b2(x, m)

        # Pool: 13 -> 6
        x, m = self.pool1(x, m)

        # Stage 2 (6x6)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)

        # Pool: 6 -> 3
        x, m = self.pool2(x, m)

        # Bottleneck (3x3)
        x, m = self.b5(x, m)

        h = self.gap(x, m)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1.")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels})."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE ({PATCH_SIZE})."
        )

    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )

In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (CPConv5 model)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_9_ens_by_logratio_pconv_stand_patch_new")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_9_cpconv5_new_patch.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 13                # <-- MUST MATCH TRAINING
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training / numeric stability
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
WRITE_LOGR = False  # False => sigmoid(log_ratio) in (0,1)

# Ensemble mode
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — UPDATED TO CPConv5 (channel-aware input pconv)
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class ChannelwisePartialConv2d(nn.Module):
    """
    Channel-aware partial conv for the *input covariate* layer.
    Uses (B,C,H,W) mask without collapsing channels.

    Per-channel normalization inside the kernel, then standard Conv2d mixes channels.
    Returns y and a spatial mask (B,1,H,W) showing any covariate support.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)

        w = torch.ones(in_channels, 1, kH, kW)
        self.register_buffer("mask_weight", w)

        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x,m: (B,C,H,W)
        if m is None:
            m = torch.ones_like(x)
        if x.dim() != 4 or m.dim() != 4:
            raise ValueError(f"Expected x,m 4D. Got x={x.shape}, m={m.shape}")
        if m.shape != x.shape:
            raise ValueError(f"Channelwise layer expects mask same shape as x. Got x={x.shape}, m={m.shape}")

        m = (m > 0).to(dtype=x.dtype)

        # Per-channel valid counts: (B,C,H,W)
        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))

        scale = self.kernel_area / (m_sum + self.eps)
        x_norm = x * m * scale
        x_norm = x_norm * (m_sum > 0).to(dtype=x.dtype)

        y = self.conv(x_norm)

        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(dtype=x.dtype)  # (B,1,H,W)

        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    """Partial conv in feature space with a 1-channel spatial mask (B,1,H,W)."""
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        if m.dim() != 4 or m.size(1) != 1:
            raise ValueError(f"Spatial mask must be (B,1,H,W). Got {m.shape}")

        m = (m > 0).to(dtype=x.dtype)
        x_masked = x * m
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride, padding=self.padding)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    """First block: channel-aware partial conv (input covariates)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)  # m is spatial (B,1,H,W)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class CPConvBlock(nn.Module):
    """Later blocks: spatial partial conv in feature space."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = SpatialPartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.p(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class PatchEncoderCPConv5(nn.Module):
    """
    Patch-size agnostic encoder (no pooling, AdaptiveAvgPool2d(1)),
    channel-aware at input via ChannelwisePartialConv2d.
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.b3 = CPConvBlock(16, 32)
        self.b4 = CPConvBlock(32, 32)
        self.b5 = CPConvBlock(32, 32)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,S,S), x_mask: (B,C,S,S)
        m_cov = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m_cov)  # spatial mask (B,1,S,S)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)
        h = self.gap(x)      # (B,32,1,1)
        z = self.proj(h)     # (B,emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoderCPConv5(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain pi0/pi1, arch params
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1, arch params)
      - fold_ids.npy
      - fold_weights_loss.npy
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels (values + per-channel masks)
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (CPConv5 architecture)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians (for imputation)
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)   # (B,C,S,S)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)  # (B,C,S,S)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # Output selection
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        out_map = sigmoid_np(log_r_map).astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_stand")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask: (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    # Apply noise ONLY on valid pixels
    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        m = (mask > 0).to(dtype=values.dtype)
        values = values + noise * m

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# NEW: Channel-aware input partial conv + spatial partial conv
# =========================================================
class ChannelwisePartialConv2d(nn.Module):
    """
    Channel-aware partial conv for the *input covariate* layer.
    Fixes the "any-channel-valid" mask collapse by using (B,C,H,W) masks.

    For each input channel c:
      m_sum_c = sum(mask_c under kernel)
      x_norm_c = x_c * mask_c * (kernel_area / (m_sum_c + eps))
      x_norm_c = 0 where m_sum_c == 0

    After that, a standard Conv2d mixes channels.
    Returns y and a spatial mask m_spatial (B,1,H,W) indicating any covariate support.
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)

        w = torch.ones(in_channels, 1, kH, kW)
        self.register_buffer("mask_weight", w)

        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x,m: (B,C,H,W)
        if m is None:
            m = torch.ones_like(x)
        if x.dim() != 4 or m.dim() != 4:
            raise ValueError(f"Expected x,m 4D. Got x={x.shape}, m={m.shape}")
        if m.shape != x.shape:
            raise ValueError(f"Channelwise layer expects mask same shape as x. Got x={x.shape}, m={m.shape}")

        m = (m > 0).to(dtype=x.dtype)

        # (B,C,H,W): per-channel valid counts
        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))

        scale = self.kernel_area / (m_sum + self.eps)
        x_norm = x * m * scale
        x_norm = x_norm * (m_sum > 0).to(dtype=x.dtype)

        y = self.conv(x_norm)

        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(dtype=x.dtype)  # (B,1,H,W)

        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    """
    Standard partial conv in feature space with a 1-channel spatial mask (B,1,H,W).
    """
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        # x: (B,C,H,W), m: (B,1,H,W)
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        if m.dim() != 4 or m.size(1) != 1:
            raise ValueError(f"Spatial mask must be (B,1,H,W). Got {m.shape}")

        m = (m > 0).to(dtype=x.dtype)
        x_masked = x * m
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride, padding=self.padding)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    """First block: channel-aware partial conv (input covariates)."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)  # m is spatial (B,1,H,W)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class CPConvBlock(nn.Module):
    """Later blocks: spatial partial conv in feature space."""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = SpatialPartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.p(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


# =========================================================
# UPDATED ENCODER: channel-aware at input, patch-size agnostic
# =========================================================
class PatchEncoderCPConv5(nn.Module):
    """
    Patch-size agnostic encoder for multi-S experiments:
      - No pooling
      - AdaptiveAvgPool2d(1) at end
      - Fixes channel-mask collapse at input via ChannelwisePartialConv2d
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.b3 = CPConvBlock(16, 32)
        self.b4 = CPConvBlock(32, 32)
        self.b5 = CPConvBlock(32, 32)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,S,S), x_mask: (B,C,S,S)
        m_cov = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m_cov)  # m becomes spatial (B,1,S,S)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x)          # (B,32,1,1)
        z = self.proj(h)         # (B,emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoderCPConv5(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_stand_auc")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask: (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    # Apply noise ONLY on valid pixels
    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        m = (mask > 0).to(dtype=values.dtype)
        values = values + noise * m

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# NEW: Channel-aware input partial conv + spatial partial conv
# =========================================================
class ChannelwisePartialConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)

        w = torch.ones(in_channels, 1, kH, kW)
        self.register_buffer("mask_weight", w)

        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        if m is None:
            m = torch.ones_like(x)
        if x.dim() != 4 or m.dim() != 4:
            raise ValueError(f"Expected x,m 4D. Got x={x.shape}, m={m.shape}")
        if m.shape != x.shape:
            raise ValueError(f"Channelwise layer expects mask same shape as x. Got x={x.shape}, m={m.shape}")

        m = (m > 0).to(dtype=x.dtype)

        m_sum = F.conv2d(m, self.mask_weight, stride=self.stride, padding=self.padding, groups=x.size(1))
        scale = self.kernel_area / (m_sum + self.eps)

        x_norm = x * m * scale
        x_norm = x_norm * (m_sum > 0).to(dtype=x.dtype)

        y = self.conv(x_norm)

        with torch.no_grad():
            m_spatial = (m_sum.sum(dim=1, keepdim=True) > 0).to(dtype=x.dtype)

        return y, m_spatial


class SpatialPartialConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False, eps=1e-8):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride=stride, padding=padding, bias=bias)
        self.eps = eps
        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.kernel_area = float(kH * kW)
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.stride = stride
        self.padding = padding

    def forward(self, x, m):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)
        if m.dim() != 4 or m.size(1) != 1:
            raise ValueError(f"Spatial mask must be (B,1,H,W). Got {m.shape}")

        m = (m > 0).to(dtype=x.dtype)
        x_masked = x * m
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(m, self.mask_kernel, stride=self.stride, padding=self.padding)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class CPConvBlock1(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = ChannelwisePartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m_cov):
        x, m = self.p(x, m_cov)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class CPConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.p = SpatialPartialConv2d(in_ch, out_ch, 3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.p(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class PatchEncoderCPConv5(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = CPConvBlock1(in_value_channels, 16)
        self.b2 = CPConvBlock(16, 16)
        self.b3 = CPConvBlock(16, 32)
        self.b4 = CPConvBlock(32, 32)
        self.b5 = CPConvBlock(32, 32)

        self.gap = nn.AdaptiveAvgPool2d(1)
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m_cov = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m_cov)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoderCPConv5(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- AUC-based weights -----------
    # Use per-fold AUCs as weights (clipped to be non-negative), then normalize.
    auc_arr = np.asarray(fold_aucs, dtype=float)
    if (not np.isfinite(auc_arr).all()) or auc_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] AUCs invalid; using equal weights.")
    else:
        # shift so that random=0.5 -> weight ~0, and clip below 0
        w = np.maximum(auc_arr - 0.5, 0.0)
        if w.sum() <= 0:
            weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
            print("\n[Ensemble] AUC-based weights degenerate; using equal weights.")
        else:
            weights = w / w.sum()
            print("\n[Ensemble] AUC-based weights proportional to max(AUC-0.5,0):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), np.asarray(fold_losses, dtype=float))
    np.save(os.path.join(MODEL_DIR, "fold_aucs.npy"), auc_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_auc.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_aucs.npy, fold_weights_auc.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


# Partial convolution


## size 3


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_3_ens_by_logratio_pconv_jj")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1  # broadcast over channels
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )  # (B,1,H_out,W_out)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# UPDATED ENCODER (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6

        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,13,13)
        # x_mask: (B,C,13,13) or (B,1,13,13)
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)

        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_3_ens_by_logratio_pconv_jj")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.

    When channelwise_mask=True (used in the first layer), the mask is kept
    per-channel (B, C_in, H, W) so that missingness can differ across
    covariate channels.  The rescaling factor accounts for valid entries
    across *all* input channels within the kernel window
    (i.e. kernel_area = C_in * kH * kW).  The output mask is collapsed to a
    single spatial mask (B, 1, H_out, W_out) so that subsequent layers
    operate with a unified validity map.

    When channelwise_mask=False (default, used in all later layers), the mask
    is a single spatial channel (B, 1, H, W), exactly as before.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            # Mask kernel sums over all in_channels × kH × kW entries
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
        else:
            # Single-channel spatial mask kernel
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if self.channelwise_mask:
            # ---- Channel-wise partial convolution (first layer) ----
            # m is (B, C_in, H, W) with per-channel validity
            m_float = (m > 0).to(dtype=x.dtype)

            # Zero out invalid entries per channel
            x_masked = x * m_float

            # Standard convolution on the masked input
            y = self.conv(x_masked)

            with torch.no_grad():
                # Count valid entries across all C_in channels and the
                # spatial kernel window at each output position.
                # mask_kernel shape: (1, C_in, kH, kW) → output (B, 1, H_out, W_out)
                m_sum = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding, dilation=self.dilation
                )  # (B, 1, H_out, W_out)
                m_out = (m_sum > 0).to(dtype=x.dtype)

            # Rescale: kernel_area = C_in * kH * kW
            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale  # broadcasts (B,1,H,W) over (B,C_out,H,W)
            y = y * m_out

            # Output mask is single spatial channel (B, 1, H_out, W_out)
            # — subsequent layers use a unified spatial mask
            return y, m_out

        else:
            # ---- Standard spatial partial convolution (later layers) ----
            # Collapse to (B,1,H,W) if needed
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1  # broadcast over channels
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding, dilation=self.dilation
                )  # (B,1,H_out,W_out)
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Additional building blocks for the improved encoder
# =========================================================
class MaskedSEBlock(nn.Module):
    """
    Squeeze-and-Excitation channel attention (mask-aware).

    Computes a per-channel scaling factor from the mask-aware
    global average, letting the network learn which feature
    channels (derived from different covariates / spatial
    patterns) matter most at each sample.
    """
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.gap = MaskedGlobalAvgPool()
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        # x: (B, C, H, W),  m: (B, 1, H, W)
        s = self.gap(x, m)          # (B, C)
        s = self.fc(s)              # (B, C)
        return x * s[:, :, None, None]  # channel-wise rescale


class MaskedResBlock(nn.Module):
    """
    Residual block built from two partial-convolution layers.

    If in_ch != out_ch a 1×1 partial convolution is used on the
    skip path to match dimensions.  An optional SE block is
    applied before the residual addition.
    """
    def __init__(self, in_ch: int, out_ch: int, dilation: int = 1,
                 use_se: bool = True, se_reduction: int = 4):
        super().__init__()
        pad = dilation  # keeps spatial size unchanged for k=3
        self.conv1 = MaskedConv2d(in_ch, out_ch, kernel_size=3, stride=1,
                                  padding=pad, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = MaskedConv2d(out_ch, out_ch, kernel_size=3, stride=1,
                                  padding=1, dilation=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.act = nn.ReLU(inplace=True)

        # 1×1 projection on skip when channels change
        self.skip_proj = None
        if in_ch != out_ch:
            self.skip_proj = MaskedConv2d(in_ch, out_ch, kernel_size=1,
                                          stride=1, padding=0, bias=False)
            self.skip_bn = nn.BatchNorm2d(out_ch)

        # Optional SE attention
        self.se = MaskedSEBlock(out_ch, reduction=se_reduction) if use_se else None

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        identity = x

        out, m_out = self.conv1(x, m)
        out = self.act(self.bn1(out))

        out, m_out = self.conv2(out, m_out)
        out = self.bn2(out)

        if self.se is not None:
            out = self.se(out, m_out)

        if self.skip_proj is not None:
            identity, _ = self.skip_proj(identity, m)
            identity = self.skip_bn(identity)

        out = self.act(out + identity)
        return out, m_out


# =========================================================
# IMPROVED ENCODER — multi-scale bottleneck with mask-aware
# partial convolutions, residual connections, SE attention,
# and multi-scale feature aggregation.
#
# Architecture (for 13×13 input patches):
#
#   Stage 1  (13×13)  C_in → 32   channel-wise mask entry
#                                   + residual block
#   Pool              13  → 6
#   Stage 2  ( 6× 6)  32  → 64    residual block
#                                   + dilated residual block
#   Pool               6  → 3
#   Stage 3  ( 3× 3)  64  → 128   bottleneck residual block
#
#   Multi-scale GAP at each stage → concat (32+64+128 = 224)
#   Linear projection → emb_dim
#
# Design rationale:
#   • Progressive down-sampling builds a proper feature
#     hierarchy (local textures → broad spatial context).
#   • Residual connections improve gradient flow in the
#     deeper network.
#   • SE blocks let the network learn which covariate-derived
#     channels are most informative per sample.
#   • Multi-scale aggregation exposes the DRE head to both
#     fine-grained and coarse spatial summaries, similar to
#     how the encoder path of a U-Net captures information
#     at every resolution level.
#   • The dilated block in Stage 2 widens the receptive field
#     without further down-sampling, consistent with the
#     paper's description.
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()

        # ---------- Stage 1: 13×13, C_in → 32 ----------
        # Entry convolution with channel-wise mask
        self.entry = MaskedConvBlock(
            in_value_channels, 32, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        # Residual refinement (same resolution)
        self.stage1_res = MaskedResBlock(32, 32, dilation=1, use_se=True)

        # ---------- Pool: 13 → 6 ----------
        self.pool1 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 2: 6×6, 32 → 64 ----------
        self.stage2_res1 = MaskedResBlock(32, 64, dilation=1, use_se=True)
        # Dilated residual block — widens receptive field
        self.stage2_res2 = MaskedResBlock(64, 64, dilation=2, use_se=True)

        # ---------- Pool: 6 → 3 ----------
        self.pool2 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 3 / Bottleneck: 3×3, 64 → 128 ----------
        self.stage3_res = MaskedResBlock(64, 128, dilation=1, use_se=True)

        # ---------- Multi-scale aggregation ----------
        self.gap = MaskedGlobalAvgPool()
        # 32 (stage1) + 64 (stage2) + 128 (stage3) = 224
        agg_dim = 32 + 64 + 128
        self.proj = nn.Sequential(
            nn.Linear(agg_dim, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val: torch.Tensor, x_mask: torch.Tensor) -> torch.Tensor:
        # x_val:  (B, C, 13, 13)
        # x_mask: (B, C, 13, 13) — per-channel validity mask
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # ---- Stage 1 (13×13) ----
        x, m = self.entry(x_val, m)     # channel-wise → spatial mask
        x, m = self.stage1_res(x, m)
        s1 = self.gap(x, m)             # (B, 32)

        # ---- Pool → Stage 2 (6×6) ----
        x, m = self.pool1(x, m)
        x, m = self.stage2_res1(x, m)
        x, m = self.stage2_res2(x, m)   # dilated
        s2 = self.gap(x, m)             # (B, 64)

        # ---- Pool → Stage 3 / Bottleneck (3×3) ----
        x, m = self.pool2(x, m)
        x, m = self.stage3_res(x, m)
        s3 = self.gap(x, m)             # (B, 128)

        # ---- Multi-scale concat + projection ----
        h = torch.cat([s1, s2, s3], dim=1)  # (B, 224)
        z = self.proj(h)                     # (B, emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")

In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED FOR PCONV MODEL)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_3_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_3_pconv.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 3                 # <-- MUST MATCH TRAINING (pconv uses 13x13)
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
WRITE_LOGR = False  # False => sigmoid(log_ratio) in (0,1)

# Ensemble mode
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — UPDATED (mask-aware / partial conv)
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    Mask propagation: m_out = 1 where any valid input exists under kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))                # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain pi0/pi1, arch params
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1, arch params)
      - fold_ids.npy
      - fold_weights_loss.npy
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (fold-specific priors in each ckpt)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # Output selection
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


## Size 5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_New_13")
TEST_DIR = str(DATA_ROOT / "test_patches_New_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_13")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 200
PATIENCE = 15

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 64
HIDDEN_DIMS = [64]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.05

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1  # broadcast over channels
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )  # (B,1,H_out,W_out)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# UPDATED ENCODER (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6

        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,13,13)
        # x_mask: (B,C,13,13) or (B,1,13,13)
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)

        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


## Size 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_New_13")
TEST_DIR = str(DATA_ROOT / "test_patches_New_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_jj")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1  # broadcast over channels
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )  # (B,1,H_out,W_out)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# UPDATED ENCODER (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6

        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,13,13)
        # x_mask: (B,C,13,13) or (B,1,13,13)
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)

        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_New_13")
TEST_DIR = str(DATA_ROOT / "test_patches_New_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_jj_3")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.

    When channelwise_mask=True (used in the first layer), the mask is kept
    per-channel (B, C_in, H, W) so that missingness can differ across
    covariate channels.  The rescaling factor accounts for valid entries
    across *all* input channels within the kernel window
    (i.e. kernel_area = C_in * kH * kW).  The output mask is collapsed to a
    single spatial mask (B, 1, H_out, W_out) so that subsequent layers
    operate with a unified validity map.

    When channelwise_mask=False (default, used in all later layers), the mask
    is a single spatial channel (B, 1, H, W), exactly as before.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            # Mask kernel sums over all in_channels x kH x kW entries
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
        else:
            # Single-channel spatial mask kernel
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if self.channelwise_mask:
            # ---- Channel-wise partial convolution (first layer) ----
            m_float = (m > 0).to(dtype=x.dtype)
            x_masked = x * m_float
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding, dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out

        else:
            # ---- Standard spatial partial convolution (later layers) ----
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding, dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Additional building blocks for the improved encoder
# =========================================================
class MaskedSEBlock(nn.Module):
    """
    Squeeze-and-Excitation channel attention (mask-aware).

    Computes a per-channel scaling factor from the mask-aware
    global average, letting the network learn which feature
    channels (derived from different covariates / spatial
    patterns) matter most at each sample.
    """
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.gap = MaskedGlobalAvgPool()
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        s = self.gap(x, m)              # (B, C)
        s = self.fc(s)                  # (B, C)
        return x * s[:, :, None, None]  # channel-wise rescale


class MaskedResBlock(nn.Module):
    """
    Residual block built from two partial-convolution layers.

    If in_ch != out_ch a 1x1 partial convolution is used on the
    skip path to match dimensions.  An optional SE block is
    applied before the residual addition.
    """
    def __init__(self, in_ch: int, out_ch: int, dilation: int = 1,
                 use_se: bool = True, se_reduction: int = 4):
        super().__init__()
        pad = dilation  # keeps spatial size unchanged for k=3
        self.conv1 = MaskedConv2d(in_ch, out_ch, kernel_size=3, stride=1,
                                  padding=pad, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = MaskedConv2d(out_ch, out_ch, kernel_size=3, stride=1,
                                  padding=1, dilation=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.act = nn.ReLU(inplace=True)

        # 1x1 projection on skip when channels change
        self.skip_proj = None
        if in_ch != out_ch:
            self.skip_proj = MaskedConv2d(in_ch, out_ch, kernel_size=1,
                                          stride=1, padding=0, bias=False)
            self.skip_bn = nn.BatchNorm2d(out_ch)

        # Optional SE attention
        self.se = MaskedSEBlock(out_ch, reduction=se_reduction) if use_se else None

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        identity = x

        out, m_out = self.conv1(x, m)
        out = self.act(self.bn1(out))

        out, m_out = self.conv2(out, m_out)
        out = self.bn2(out)

        if self.se is not None:
            out = self.se(out, m_out)

        if self.skip_proj is not None:
            identity, _ = self.skip_proj(identity, m)
            identity = self.skip_bn(identity)

        out = self.act(out + identity)
        return out, m_out


# =========================================================
# IMPROVED ENCODER — multi-scale bottleneck with mask-aware
# partial convolutions, residual connections, SE attention,
# and multi-scale feature aggregation.
#
# Architecture (for 13x13 input patches):
#
#   Stage 1  (13x13)  C_in -> 32   channel-wise mask entry
#                                   + residual block
#   Pool              13   -> 6
#   Stage 2  ( 6x 6)  32  -> 64    residual block
#                                   + dilated residual block
#   Pool               6  -> 3
#   Stage 3  ( 3x 3)  64  -> 128   bottleneck residual block
#
#   Multi-scale GAP at each stage -> concat (32+64+128 = 224)
#   Linear projection -> emb_dim
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()

        # ---------- Stage 1: 13x13, C_in -> 32 ----------
        # Entry convolution with channel-wise mask
        self.entry = MaskedConvBlock(
            in_value_channels, 32, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        # Residual refinement (same resolution)
        self.stage1_res = MaskedResBlock(32, 32, dilation=1, use_se=True)

        # ---------- Pool: 13 -> 6 ----------
        self.pool1 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 2: 6x6, 32 -> 64 ----------
        self.stage2_res1 = MaskedResBlock(32, 64, dilation=1, use_se=True)
        # Dilated residual block — widens receptive field
        self.stage2_res2 = MaskedResBlock(64, 64, dilation=2, use_se=True)

        # ---------- Pool: 6 -> 3 ----------
        self.pool2 = MaskedMaxPool2d(2, 2)

        # ---------- Stage 3 / Bottleneck: 3x3, 64 -> 128 ----------
        self.stage3_res = MaskedResBlock(64, 128, dilation=1, use_se=True)

        # ---------- Multi-scale aggregation ----------
        self.gap = MaskedGlobalAvgPool()
        # 32 (stage1) + 64 (stage2) + 128 (stage3) = 224
        agg_dim = 32 + 64 + 128
        self.proj = nn.Sequential(
            nn.Linear(agg_dim, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val: torch.Tensor, x_mask: torch.Tensor) -> torch.Tensor:
        # x_val:  (B, C, 13, 13)
        # x_mask: (B, C, 13, 13) — per-channel validity mask
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # ---- Stage 1 (13x13) ----
        x, m = self.entry(x_val, m)     # channel-wise -> spatial mask
        x, m = self.stage1_res(x, m)
        s1 = self.gap(x, m)             # (B, 32)

        # ---- Pool -> Stage 2 (6x6) ----
        x, m = self.pool1(x, m)
        x, m = self.stage2_res1(x, m)
        x, m = self.stage2_res2(x, m)   # dilated
        s2 = self.gap(x, m)             # (B, 64)

        # ---- Pool -> Stage 3 / Bottleneck (3x3) ----
        x, m = self.pool2(x, m)
        x, m = self.stage3_res(x, m)
        s3 = self.gap(x, m)             # (B, 128)

        # ---- Multi-scale concat + projection ----
        h = torch.cat([s1, s2, s3], dim=1)  # (B, 224)
        z = self.proj(h)                     # (B, emb_dim)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")

In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED FOR PCONV MODEL)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_origin")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_13_pconv_origin.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 13                 # <-- MUST MATCH TRAINING (pconv uses 13x13)
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
WRITE_LOGR = False  # False => sigmoid(log_ratio) in (0,1)

# Ensemble mode
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — UPDATED (mask-aware / partial conv)
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    Mask propagation: m_out = 1 where any valid input exists under kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))                # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain pi0/pi1, arch params
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1, arch params)
      - fold_ids.npy
      - fold_weights_loss.npy
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (fold-specific priors in each ckpt)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # Output selection
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED FOR IMPROVED PCONV MODEL)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pcon_jj")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_13_pconv_jj") + ".tif"

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 13                 # <-- MUST MATCH TRAINING (pconv uses 13x13)
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
WRITE_LOGR = False  # False => sigmoid(log_ratio) in (0,1)

# Ensemble mode
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — IMPROVED (mask-aware / partial conv)
# with channel-wise masking, residual blocks, SE attention,
# dilated convolutions, and multi-scale feature aggregation.
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    Mask propagation: m_out = 1 where any valid input exists under kernel.

    When channelwise_mask=True (first layer only), the mask is kept
    per-channel (B, C_in, H, W) and rescaling accounts for all
    C_in * kH * kW entries.  The output mask is collapsed to a
    single spatial mask (B, 1, H_out, W_out).
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
        channelwise_mask: bool = False,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps
        self.channelwise_mask = channelwise_mask

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size

        if channelwise_mask:
            self.register_buffer("mask_kernel", torch.ones(1, in_channels, kH, kW))
            self.kernel_area = float(in_channels * kH * kW)
        else:
            self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
            self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones_like(x)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        if self.channelwise_mask:
            m_float = (m > 0).to(dtype=x.dtype)
            x_masked = x * m_float
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m_float, self.mask_kernel,
                    stride=self.stride, padding=self.padding, dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out

        else:
            if m.size(1) != 1:
                m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
            else:
                m1 = (m > 0).to(dtype=x.dtype)

            x_masked = x * m1
            y = self.conv(x_masked)

            with torch.no_grad():
                m_sum = F.conv2d(
                    m1, self.mask_kernel,
                    stride=self.stride, padding=self.padding, dilation=self.dilation
                )
                m_out = (m_sum > 0).to(dtype=x.dtype)

            scale = self.kernel_area / (m_sum + self.eps)
            y = y * scale
            y = y * m_out
            return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1, dilation=1, channelwise_mask=False):
        super().__init__()
        self.mconv = MaskedConv2d(
            in_ch, out_ch, kernel_size=k, stride=s, padding=p,
            dilation=dilation, bias=False, channelwise_mask=channelwise_mask,
        )
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))                # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


class MaskedSEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention (mask-aware)."""
    def __init__(self, channels: int, reduction: int = 4):
        super().__init__()
        mid = max(channels // reduction, 4)
        self.gap = MaskedGlobalAvgPool()
        self.fc = nn.Sequential(
            nn.Linear(channels, mid),
            nn.ReLU(inplace=True),
            nn.Linear(mid, channels),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor, m: torch.Tensor) -> torch.Tensor:
        s = self.gap(x, m)
        s = self.fc(s)
        return x * s[:, :, None, None]


class MaskedResBlock(nn.Module):
    """
    Residual block with two partial-convolution layers.
    1x1 projection on the skip path when channels change.
    Optional SE attention before residual addition.
    """
    def __init__(self, in_ch: int, out_ch: int, dilation: int = 1,
                 use_se: bool = True, se_reduction: int = 4):
        super().__init__()
        pad = dilation
        self.conv1 = MaskedConv2d(in_ch, out_ch, kernel_size=3, stride=1,
                                  padding=pad, dilation=dilation, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)

        self.conv2 = MaskedConv2d(out_ch, out_ch, kernel_size=3, stride=1,
                                  padding=1, dilation=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.act = nn.ReLU(inplace=True)

        self.skip_proj = None
        if in_ch != out_ch:
            self.skip_proj = MaskedConv2d(in_ch, out_ch, kernel_size=1,
                                          stride=1, padding=0, bias=False)
            self.skip_bn = nn.BatchNorm2d(out_ch)

        self.se = MaskedSEBlock(out_ch, reduction=se_reduction) if use_se else None

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        identity = x

        out, m_out = self.conv1(x, m)
        out = self.act(self.bn1(out))

        out, m_out = self.conv2(out, m_out)
        out = self.bn2(out)

        if self.se is not None:
            out = self.se(out, m_out)

        if self.skip_proj is not None:
            identity, _ = self.skip_proj(identity, m)
            identity = self.skip_bn(identity)

        out = self.act(out + identity)
        return out, m_out


class PatchEncoder13(nn.Module):
    """
    Multi-scale bottleneck encoder with mask-aware partial
    convolutions, residual connections, SE attention, and
    multi-scale feature aggregation.

    Stage 1  (13x13)  C_in -> 32   channel-wise mask entry + res block
    Pool              13   -> 6
    Stage 2  ( 6x 6)  32  -> 64    res block + dilated res block
    Pool               6  -> 3
    Stage 3  ( 3x 3)  64  -> 128   bottleneck res block

    Multi-scale GAP -> concat (32+64+128=224) -> proj -> emb_dim
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()

        # Stage 1: 13x13, C_in -> 32
        self.entry = MaskedConvBlock(
            in_value_channels, 32, k=3, s=1, p=1,
            channelwise_mask=True,
        )
        self.stage1_res = MaskedResBlock(32, 32, dilation=1, use_se=True)

        # Pool: 13 -> 6
        self.pool1 = MaskedMaxPool2d(2, 2)

        # Stage 2: 6x6, 32 -> 64
        self.stage2_res1 = MaskedResBlock(32, 64, dilation=1, use_se=True)
        self.stage2_res2 = MaskedResBlock(64, 64, dilation=2, use_se=True)

        # Pool: 6 -> 3
        self.pool2 = MaskedMaxPool2d(2, 2)

        # Stage 3 / Bottleneck: 3x3, 64 -> 128
        self.stage3_res = MaskedResBlock(64, 128, dilation=1, use_se=True)

        # Multi-scale aggregation
        self.gap = MaskedGlobalAvgPool()
        agg_dim = 32 + 64 + 128  # 224
        self.proj = nn.Sequential(
            nn.Linear(agg_dim, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val: torch.Tensor, x_mask: torch.Tensor) -> torch.Tensor:
        m = (x_mask > 0).to(dtype=x_val.dtype)

        # Stage 1 (13x13)
        x, m = self.entry(x_val, m)
        x, m = self.stage1_res(x, m)
        s1 = self.gap(x, m)

        # Pool -> Stage 2 (6x6)
        x, m = self.pool1(x, m)
        x, m = self.stage2_res1(x, m)
        x, m = self.stage2_res2(x, m)
        s2 = self.gap(x, m)

        # Pool -> Stage 3 / Bottleneck (3x3)
        x, m = self.pool2(x, m)
        x, m = self.stage3_res(x, m)
        s3 = self.gap(x, m)

        # Multi-scale concat + projection
        h = torch.cat([s1, s2, s3], dim=1)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain pi0/pi1, arch params
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1, arch params)
      - fold_ids.npy
      - fold_weights_loss.npy
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (fold-specific priors in each ckpt)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # Output selection
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )

## Size 23


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_23_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 23  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1  # broadcast over channels
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )  # (B,1,H_out,W_out)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# UPDATED ENCODER (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6

        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,13,13)
        # x_mask: (B,C,13,13) or (B,1,13,13)
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)

        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED FOR PCONV MODEL)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_23_ens_by_logratio_pconv")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_23_pconv.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 23                 # <-- MUST MATCH TRAINING (pconv uses 13x13)
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
WRITE_LOGR = False  # False => sigmoid(log_ratio) in (0,1)

# Ensemble mode
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# =============================================================
# Model definitions — UPDATED (mask-aware / partial conv)
# =============================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    Mask propagation: m_out = 1 where any valid input exists under kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))                # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps)  # (B,1)
        return num / den


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)
        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)
        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain pi0/pi1, arch params
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1, arch params)
      - fold_ids.npy
      - fold_weights_loss.npy
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)
    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (fold-specific priors in each ckpt)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)

    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # Output selection
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


## size 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_33_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1  # broadcast over channels
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )  # (B,1,H_out,W_out)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# UPDATED ENCODER (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6

        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,13,13)
        # x_mask: (B,C,13,13) or (B,1,13,13)
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)

        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


## Size 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_65_ens_by_logratio_pconv")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Mask-aware / Partial convolution blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W) if needed
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1  # broadcast over channels
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )  # (B,1,H_out,W_out)
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedMaxPool2d(nn.Module):
    def __init__(self, kernel_size=2, stride=2):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size, stride)

    def forward(self, x, m):
        x = self.pool(x)
        m = self.pool(m)
        m = (m > 0).to(dtype=x.dtype)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# UPDATED ENCODER (mask-aware)
# =========================================================
class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.pool = MaskedMaxPool2d(2, 2)  # 13 -> 6

        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()

        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        # x_val: (B,C,13,13)
        # x_mask: (B,C,13,13) or (B,1,13,13)
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.pool(x, m)
        x, m = self.b3(x, m)

        h = self.gap(x, m)  # (B,32)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


## partial conv opt 2


#### Size 3


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_3_ens_by_logratio_pconv_2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# PATCH_SIZE is inferred from X arrays (so you can run S=3,5,9,13,23,33,65,...)
PATCH_SIZE = None

# Choose encoder:
#   "pconv" -> partial / mask-aware conv encoder (patch-size-agnostic)
#   "std"   -> standard conv encoder (patch-size-agnostic)
ENCODER_TYPE = "pconv"  # change to "std" for standard encoder baseline

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Helpers: patch size inference (for multi-size experiments)
# =========================================================
def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask here are (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])  # flip H
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])  # flip W
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


# =========================================================
# Head
# =========================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Partial / mask-aware conv blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W)
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        # keeping BN here (as in your original pconv code). If you want BN removed for fairness, say so.
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Encoders
# =========================================================
class PatchEncoderPConv5(nn.Module):
    """
    Partial-conv encoder, patch-size-agnostic:
    - NO pooling/strides (fair across S)
    - masked global average pooling
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=1)
        self.b5 = MaskedConvBlock(32, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x, m)  # (B,32)
        return self.proj(h)


class PatchEncoderStd5Conv(nn.Module):
    """
    Standard encoder for fair patch-size comparison:
    - mask gates values: x_val * x_mask
    - concatenates [masked values, mask] as channels (2C)
    - 5 conv blocks, no pooling, patch-size-agnostic, global avg pool
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


# =========================================================
# Model
# =========================================================
class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        if ENCODER_TYPE == "pconv":
            self.encoder = PatchEncoderPConv5(in_value_channels, emb_dim)
        elif ENCODER_TYPE == "std":
            self.encoder = PatchEncoderStd5Conv(in_value_channels, emb_dim)
        else:
            raise ValueError(f"Unknown ENCODER_TYPE={ENCODER_TYPE}. Use 'pconv' or 'std'.")

        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s} | encoder={ENCODER_TYPE}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(patch_s),
                "encoder_type": str(ENCODER_TYPE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if state.get("encoder_type", None) != ENCODER_TYPE:
            raise RuntimeError(f"Encoder type mismatch for fold {fid}: ckpt={state.get('encoder_type')} vs run={ENCODER_TYPE}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)
    print("[CV] inferred patch size:", infer_patch_s(X_cv))

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)
    print("[TEST] inferred patch size:", infer_patch_s(X_test))

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    if infer_patch_s(X_test) != infer_patch_s(X_cv):
        raise ValueError("CV and TEST patch sizes differ. Use matching extracted patch datasets.")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


#### Size 5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_3_ens_by_logratio_pconv_2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# PATCH_SIZE is inferred from X arrays (so you can run S=3,5,9,13,23,33,65,...)
PATCH_SIZE = None

# Choose encoder:
#   "pconv" -> partial / mask-aware conv encoder (patch-size-agnostic)
#   "std"   -> standard conv encoder (patch-size-agnostic)
ENCODER_TYPE = "pconv"  # change to "std" for standard encoder baseline

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Helpers: patch size inference (for multi-size experiments)
# =========================================================
def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask here are (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])  # flip H
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])  # flip W
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


# =========================================================
# Head
# =========================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Partial / mask-aware conv blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W)
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        # keeping BN here (as in your original pconv code). If you want BN removed for fairness, say so.
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Encoders
# =========================================================
class PatchEncoderPConv5(nn.Module):
    """
    Partial-conv encoder, patch-size-agnostic:
    - NO pooling/strides (fair across S)
    - masked global average pooling
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=1)
        self.b5 = MaskedConvBlock(32, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x, m)  # (B,32)
        return self.proj(h)


class PatchEncoderStd5Conv(nn.Module):
    """
    Standard encoder for fair patch-size comparison:
    - mask gates values: x_val * x_mask
    - concatenates [masked values, mask] as channels (2C)
    - 5 conv blocks, no pooling, patch-size-agnostic, global avg pool
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


# =========================================================
# Model
# =========================================================
class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        if ENCODER_TYPE == "pconv":
            self.encoder = PatchEncoderPConv5(in_value_channels, emb_dim)
        elif ENCODER_TYPE == "std":
            self.encoder = PatchEncoderStd5Conv(in_value_channels, emb_dim)
        else:
            raise ValueError(f"Unknown ENCODER_TYPE={ENCODER_TYPE}. Use 'pconv' or 'std'.")

        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s} | encoder={ENCODER_TYPE}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(patch_s),
                "encoder_type": str(ENCODER_TYPE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if state.get("encoder_type", None) != ENCODER_TYPE:
            raise RuntimeError(f"Encoder type mismatch for fold {fid}: ckpt={state.get('encoder_type')} vs run={ENCODER_TYPE}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)
    print("[CV] inferred patch size:", infer_patch_s(X_cv))

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)
    print("[TEST] inferred patch size:", infer_patch_s(X_test))

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    if infer_patch_s(X_test) != infer_patch_s(X_cv):
        raise ValueError("CV and TEST patch sizes differ. Use matching extracted patch datasets.")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


#### Size 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_13_ens_by_logratio_pconv_2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# PATCH_SIZE is inferred from X arrays (so you can run S=3,5,9,13,23,33,65,...)
PATCH_SIZE = None

# Choose encoder:
#   "pconv" -> partial / mask-aware conv encoder (patch-size-agnostic)
#   "std"   -> standard conv encoder (patch-size-agnostic)
ENCODER_TYPE = "pconv"  # change to "std" for standard encoder baseline

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Helpers: patch size inference (for multi-size experiments)
# =========================================================
def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask here are (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])  # flip H
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])  # flip W
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


# =========================================================
# Head
# =========================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Partial / mask-aware conv blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W)
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        # keeping BN here (as in your original pconv code). If you want BN removed for fairness, say so.
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Encoders
# =========================================================
class PatchEncoderPConv5(nn.Module):
    """
    Partial-conv encoder, patch-size-agnostic:
    - NO pooling/strides (fair across S)
    - masked global average pooling
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=1)
        self.b5 = MaskedConvBlock(32, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x, m)  # (B,32)
        return self.proj(h)


class PatchEncoderStd5Conv(nn.Module):
    """
    Standard encoder for fair patch-size comparison:
    - mask gates values: x_val * x_mask
    - concatenates [masked values, mask] as channels (2C)
    - 5 conv blocks, no pooling, patch-size-agnostic, global avg pool
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


# =========================================================
# Model
# =========================================================
class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        if ENCODER_TYPE == "pconv":
            self.encoder = PatchEncoderPConv5(in_value_channels, emb_dim)
        elif ENCODER_TYPE == "std":
            self.encoder = PatchEncoderStd5Conv(in_value_channels, emb_dim)
        else:
            raise ValueError(f"Unknown ENCODER_TYPE={ENCODER_TYPE}. Use 'pconv' or 'std'.")

        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s} | encoder={ENCODER_TYPE}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(patch_s),
                "encoder_type": str(ENCODER_TYPE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if state.get("encoder_type", None) != ENCODER_TYPE:
            raise RuntimeError(f"Encoder type mismatch for fold {fid}: ckpt={state.get('encoder_type')} vs run={ENCODER_TYPE}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)
    print("[CV] inferred patch size:", infer_patch_s(X_cv))

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)
    print("[TEST] inferred patch size:", infer_patch_s(X_test))

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    if infer_patch_s(X_test) != infer_patch_s(X_cv):
        raise ValueError("CV and TEST patch sizes differ. Use matching extracted patch datasets.")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


#### Size 23


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_23_ens_by_logratio_pconv_2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# PATCH_SIZE is inferred from X arrays (so you can run S=3,5,9,13,23,33,65,...)
PATCH_SIZE = None

# Choose encoder:
#   "pconv" -> partial / mask-aware conv encoder (patch-size-agnostic)
#   "std"   -> standard conv encoder (patch-size-agnostic)
ENCODER_TYPE = "pconv"  # change to "std" for standard encoder baseline

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Helpers: patch size inference (for multi-size experiments)
# =========================================================
def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask here are (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])  # flip H
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])  # flip W
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


# =========================================================
# Head
# =========================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Partial / mask-aware conv blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W)
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        # keeping BN here (as in your original pconv code). If you want BN removed for fairness, say so.
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Encoders
# =========================================================
class PatchEncoderPConv5(nn.Module):
    """
    Partial-conv encoder, patch-size-agnostic:
    - NO pooling/strides (fair across S)
    - masked global average pooling
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=1)
        self.b5 = MaskedConvBlock(32, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x, m)  # (B,32)
        return self.proj(h)


class PatchEncoderStd5Conv(nn.Module):
    """
    Standard encoder for fair patch-size comparison:
    - mask gates values: x_val * x_mask
    - concatenates [masked values, mask] as channels (2C)
    - 5 conv blocks, no pooling, patch-size-agnostic, global avg pool
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


# =========================================================
# Model
# =========================================================
class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        if ENCODER_TYPE == "pconv":
            self.encoder = PatchEncoderPConv5(in_value_channels, emb_dim)
        elif ENCODER_TYPE == "std":
            self.encoder = PatchEncoderStd5Conv(in_value_channels, emb_dim)
        else:
            raise ValueError(f"Unknown ENCODER_TYPE={ENCODER_TYPE}. Use 'pconv' or 'std'.")

        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s} | encoder={ENCODER_TYPE}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(patch_s),
                "encoder_type": str(ENCODER_TYPE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if state.get("encoder_type", None) != ENCODER_TYPE:
            raise RuntimeError(f"Encoder type mismatch for fold {fid}: ckpt={state.get('encoder_type')} vs run={ENCODER_TYPE}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)
    print("[CV] inferred patch size:", infer_patch_s(X_cv))

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)
    print("[TEST] inferred patch size:", infer_patch_s(X_test))

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    if infer_patch_s(X_test) != infer_patch_s(X_cv):
        raise ValueError("CV and TEST patch sizes differ. Use matching extracted patch datasets.")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


#### Size 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_33_ens_by_logratio_pconv_2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# PATCH_SIZE is inferred from X arrays (so you can run S=3,5,9,13,23,33,65,...)
PATCH_SIZE = None

# Choose encoder:
#   "pconv" -> partial / mask-aware conv encoder (patch-size-agnostic)
#   "std"   -> standard conv encoder (patch-size-agnostic)
ENCODER_TYPE = "pconv"  # change to "std" for standard encoder baseline

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Helpers: patch size inference (for multi-size experiments)
# =========================================================
def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask here are (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])  # flip H
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])  # flip W
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


# =========================================================
# Head
# =========================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Partial / mask-aware conv blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W)
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        # keeping BN here (as in your original pconv code). If you want BN removed for fairness, say so.
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Encoders
# =========================================================
class PatchEncoderPConv5(nn.Module):
    """
    Partial-conv encoder, patch-size-agnostic:
    - NO pooling/strides (fair across S)
    - masked global average pooling
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=1)
        self.b5 = MaskedConvBlock(32, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x, m)  # (B,32)
        return self.proj(h)


class PatchEncoderStd5Conv(nn.Module):
    """
    Standard encoder for fair patch-size comparison:
    - mask gates values: x_val * x_mask
    - concatenates [masked values, mask] as channels (2C)
    - 5 conv blocks, no pooling, patch-size-agnostic, global avg pool
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


# =========================================================
# Model
# =========================================================
class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        if ENCODER_TYPE == "pconv":
            self.encoder = PatchEncoderPConv5(in_value_channels, emb_dim)
        elif ENCODER_TYPE == "std":
            self.encoder = PatchEncoderStd5Conv(in_value_channels, emb_dim)
        else:
            raise ValueError(f"Unknown ENCODER_TYPE={ENCODER_TYPE}. Use 'pconv' or 'std'.")

        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s} | encoder={ENCODER_TYPE}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(patch_s),
                "encoder_type": str(ENCODER_TYPE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if state.get("encoder_type", None) != ENCODER_TYPE:
            raise RuntimeError(f"Encoder type mismatch for fold {fid}: ckpt={state.get('encoder_type')} vs run={ENCODER_TYPE}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)
    print("[CV] inferred patch size:", infer_patch_s(X_cv))

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)
    print("[TEST] inferred patch size:", infer_patch_s(X_test))

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    if infer_patch_s(X_test) != infer_patch_s(X_cv):
        raise ValueError("CV and TEST patch sizes differ. Use matching extracted patch datasets.")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


#### Size 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_models_65_ens_by_logratio_pconv_2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# PATCH_SIZE is inferred from X arrays (so you can run S=3,5,9,13,23,33,65,...)
PATCH_SIZE = None

# Choose encoder:
#   "pconv" -> partial / mask-aware conv encoder (patch-size-agnostic)
#   "std"   -> standard conv encoder (patch-size-agnostic)
ENCODER_TYPE = "pconv"  # change to "std" for standard encoder baseline

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Helpers: patch size inference (for multi-size experiments)
# =========================================================
def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    # values/mask here are (C,H,W)
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])  # flip H
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])  # flip W
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


# =========================================================
# Head
# =========================================================
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


# =========================================================
# Partial / mask-aware conv blocks
# =========================================================
class MaskedConv2d(nn.Module):
    """
    Partial / mask-aware convolution:
      y = conv(x * m) * (kernel_area / (sum(m under kernel)+eps))
      y = 0 where sum(m under kernel)==0
    The mask is propagated: m_out = 1 where any valid input exists under the kernel.
    """
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int = 3,
        stride: int = 1,
        padding: int = 1,
        dilation: int = 1,
        bias: bool = False,
        eps: float = 1e-8,
    ):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels, out_channels,
            kernel_size=kernel_size, stride=stride, padding=padding, dilation=dilation,
            bias=bias
        )
        self.eps = eps

        kH, kW = (kernel_size, kernel_size) if isinstance(kernel_size, int) else kernel_size
        self.register_buffer("mask_kernel", torch.ones(1, 1, kH, kW))
        self.kernel_area = float(kH * kW)

        self.stride = stride
        self.padding = padding
        self.dilation = dilation

    def forward(self, x: torch.Tensor, m: torch.Tensor):
        if m is None:
            m = torch.ones(x.size(0), 1, x.size(2), x.size(3), device=x.device, dtype=x.dtype)

        if m.dim() != 4:
            raise ValueError(f"Mask must be 4D (B,C,H,W or B,1,H,W). Got {m.shape}")

        # collapse to (B,1,H,W)
        if m.size(1) != 1:
            m1 = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m1 = (m > 0).to(dtype=x.dtype)

        x_masked = x * m1
        y = self.conv(x_masked)

        with torch.no_grad():
            m_sum = F.conv2d(
                m1, self.mask_kernel,
                stride=self.stride, padding=self.padding, dilation=self.dilation
            )
            m_out = (m_sum > 0).to(dtype=x.dtype)

        scale = self.kernel_area / (m_sum + self.eps)
        y = y * scale
        y = y * m_out
        return y, m_out


class MaskedConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, k=3, s=1, p=1):
        super().__init__()
        self.mconv = MaskedConv2d(in_ch, out_ch, kernel_size=k, stride=s, padding=p, bias=False)
        # keeping BN here (as in your original pconv code). If you want BN removed for fairness, say so.
        self.bn = nn.BatchNorm2d(out_ch)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x, m):
        x, m = self.mconv(x, m)
        x = self.bn(x)
        x = self.act(x)
        return x, m


class MaskedGlobalAvgPool(nn.Module):
    def __init__(self, eps=1e-8):
        super().__init__()
        self.eps = eps

    def forward(self, x, m):
        if m.size(1) != 1:
            m = (m > 0).any(dim=1, keepdim=True).to(dtype=x.dtype)
        else:
            m = (m > 0).to(dtype=x.dtype)
        num = (x * m).sum(dim=(2, 3))               # (B,C)
        den = m.sum(dim=(2, 3)).clamp_min(self.eps) # (B,1)
        return num / den


# =========================================================
# Encoders
# =========================================================
class PatchEncoderPConv5(nn.Module):
    """
    Partial-conv encoder, patch-size-agnostic:
    - NO pooling/strides (fair across S)
    - masked global average pooling
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        self.b1 = MaskedConvBlock(in_value_channels, 16, k=3, s=1, p=1)
        self.b2 = MaskedConvBlock(16, 16, k=3, s=1, p=1)
        self.b3 = MaskedConvBlock(16, 32, k=3, s=1, p=1)
        self.b4 = MaskedConvBlock(32, 32, k=3, s=1, p=1)
        self.b5 = MaskedConvBlock(32, 32, k=3, s=1, p=1)

        self.gap = MaskedGlobalAvgPool()
        self.proj = nn.Sequential(
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        m = (x_mask > 0).to(dtype=x_val.dtype)

        x, m = self.b1(x_val, m)
        x, m = self.b2(x, m)
        x, m = self.b3(x, m)
        x, m = self.b4(x, m)
        x, m = self.b5(x, m)

        h = self.gap(x, m)  # (B,32)
        return self.proj(h)


class PatchEncoderStd5Conv(nn.Module):
    """
    Standard encoder for fair patch-size comparison:
    - mask gates values: x_val * x_mask
    - concatenates [masked values, mask] as channels (2C)
    - 5 conv blocks, no pooling, patch-size-agnostic, global avg pool
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, 3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


# =========================================================
# Model
# =========================================================
class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        if ENCODER_TYPE == "pconv":
            self.encoder = PatchEncoderPConv5(in_value_channels, emb_dim)
        elif ENCODER_TYPE == "std":
            self.encoder = PatchEncoderStd5Conv(in_value_channels, emb_dim)
        else:
            raise ValueError(f"Unknown ENCODER_TYPE={ENCODER_TYPE}. Use 'pconv' or 'std'.")

        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s} | encoder={ENCODER_TYPE}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": int(in_value_channels),
                "patch_size": int(patch_s),
                "encoder_type": str(ENCODER_TYPE),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if state.get("encoder_type", None) != ENCODER_TYPE:
            raise RuntimeError(f"Encoder type mismatch for fold {fid}: ckpt={state.get('encoder_type')} vs run={ENCODER_TYPE}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)
    print("[CV] inferred patch size:", infer_patch_s(X_cv))

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)
    print("[TEST] inferred patch size:", infer_patch_s(X_test))

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    if infer_patch_s(X_test) != infer_patch_s(X_cv):
        raise ValueError("CV and TEST patch sizes differ. Use matching extracted patch datasets.")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


# Patch size:3


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_3")
TEST_DIR = str(DATA_ROOT / "test_patches_3")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_classifier_3_ens_by_logratio_New")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 3  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # stable sigmoid
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # log r(x) = logit(x) + log(pi0/pi1)
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # bounded suitability in (0,1), not interpreted as prevalence probability
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    # IMPORTANT: evaluate metrics on the SAME bounded score everywhere
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# - metrics computed on bounded score sigmoid(log_ratio)
# - fold-specific priors stored in checkpoint
# - returns OOF bounded scores for that fold's val split
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score) using fold checkpoints
# - computes per-model log_ratio using stored fold priors
# - ensembles in log space (default) for stability
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                # weighted mean of log-ratios (geometric mean of ratios)
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                # ratio-space average (less stable)
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights (OK for ensembling) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_classifier_3_ens_by_logratio_New")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_3_logspace_ens2.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 3
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (unbounded, but clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain fold-specific priors (pi0/pi1)
# Training code (updated) no longer needs:
#  - class_priors.npy
#  - suitability_offset_c.npy
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1)
      - fold_ids.npy
      - fold_weights_loss.npy

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logpi_t: (K,) where logpi_t[k] = log(pi0_k/pi1_k) from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # harmless at eval (model.eval disables dropout)
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)

    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (UPDATED: fold-specific priors are in each checkpoint)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection (UPDATED)
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# patch size: 5


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_5")
TEST_DIR = str(DATA_ROOT / "test_patches_5")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_classifier_5_ens_by_logratio_New2")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

# NOTE: patch size is inferred from X arrays; this is just metadata placeholder
PATCH_SIZE = None

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins=20, min_per_group=10, min_bin_bg=10, eps=1e-9, use_log=True):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nbins + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))

    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk < min_bin_bg:
            continue

        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())

        # P/E with epsilon
        pe = ((npk + eps) / (Lp + eps)) / ((nbk + eps) / (Lb + eps))

        # representative score for the bin (more robust than midpoint)
        center = float(np.median(s_bg[in_bg]))

        pratio.append(np.log(pe) if use_log else pe)
        centers.append(center)

    if len(pratio) < 3:
        return np.nan

    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))



def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


def infer_patch_s(X_values: np.ndarray) -> int:
    if X_values.ndim != 4:
        raise ValueError(f"Expected X to have shape (N,C,S,S), got {X_values.shape}")
    h, w = int(X_values.shape[-2]), int(X_values.shape[-1])
    if h != w:
        raise ValueError(f"Expected square patches, got {h}x{w}")
    return w


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        # values/mask are (C,H,W) here
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoderAny5Conv(nn.Module):
    """
    Patch-size-agnostic encoder with 5 conv blocks.
    No BatchNorm, no pooling/strides; global average pooling at the end.
    """
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels

        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )

        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoderAny5Conv(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    patch_s = infer_patch_s(X_values)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(
        f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | "
        f"priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f} | patch={patch_s}x{patch_s}"
    )

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": int(patch_s),
                "emb_dim": int(EMB_DIM),
                "hidden_dims": list(HIDDEN_DIMS),
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score)
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]
    patch_s = infer_patch_s(X_values)

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}: ckpt={state['in_value_channels']} vs data={in_value_channels}")

        if state.get("patch_size", None) != patch_s:
            raise RuntimeError(f"Patch size mismatch for fold {fid}: ckpt={state.get('patch_size')} vs data={patch_s}")

        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            if ENSEMBLE_IN_LOGSPACE:
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # Show inferred patch size
    cv_patch_s = infer_patch_s(X_cv)
    print(f"[CV] inferred patch size: {cv_patch_s}x{cv_patch_s}")

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    test_patch_s = infer_patch_s(X_test)
    print(f"[TEST] inferred patch size: {test_patch_s}x{test_patch_s}")

    if test_patch_s != cv_patch_s:
        raise ValueError(f"CV patch size ({cv_patch_s}) and TEST patch size ({test_patch_s}) differ. Use matching datasets.")

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_5")
TEST_DIR = str(DATA_ROOT / "test_patches_5")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_5_ens_by_logratio_New")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 5  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # stable sigmoid
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # log r(x) = logit(x) + log(pi0/pi1)
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # bounded suitability in (0,1), not interpreted as prevalence probability
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    # IMPORTANT: evaluate metrics on the SAME bounded score everywhere
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# - metrics computed on bounded score sigmoid(log_ratio)
# - fold-specific priors stored in checkpoint
# - returns OOF bounded scores for that fold's val split
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score) using fold checkpoints
# - computes per-model log_ratio using stored fold priors
# - ensembles in log space (default) for stability
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                # weighted mean of log-ratios (geometric mean of ratios)
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                # ratio-space average (less stable)
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights (OK for ensembling) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_5_ens_by_logratio_New")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_5_logspace_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 5
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (unbounded, but clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain fold-specific priors (pi0/pi1)
# Training code (updated) no longer needs:
#  - class_priors.npy
#  - suitability_offset_c.npy
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1)
      - fold_ids.npy
      - fold_weights_loss.npy

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logpi_t: (K,) where logpi_t[k] = log(pi0_k/pi1_k) from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # harmless at eval (model.eval disables dropout)
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)

    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (UPDATED: fold-specific priors are in each checkpoint)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection (UPDATED)
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Patch size: 23


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_23")
TEST_DIR = str(DATA_ROOT / "test_patches_23")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_23_ens_by_logratio_New")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 23 # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # stable sigmoid
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # log r(x) = logit(x) + log(pi0/pi1)
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # bounded suitability in (0,1), not interpreted as prevalence probability
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    # IMPORTANT: evaluate metrics on the SAME bounded score everywhere
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# - metrics computed on bounded score sigmoid(log_ratio)
# - fold-specific priors stored in checkpoint
# - returns OOF bounded scores for that fold's val split
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score) using fold checkpoints
# - computes per-model log_ratio using stored fold priors
# - ensembles in log space (default) for stability
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                # weighted mean of log-ratios (geometric mean of ratios)
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                # ratio-space average (less stable)
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights (OK for ensembling) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_23_ens_by_logratio_New")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_23_logspace_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 23
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (unbounded, but clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain fold-specific priors (pi0/pi1)
# Training code (updated) no longer needs:
#  - class_priors.npy
#  - suitability_offset_c.npy
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1)
      - fold_ids.npy
      - fold_weights_loss.npy

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logpi_t: (K,) where logpi_t[k] = log(pi0_k/pi1_k) from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # harmless at eval (model.eval disables dropout)
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)

    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (UPDATED: fold-specific priors are in each checkpoint)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection (UPDATED)
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Patch size: 13


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_13")
TEST_DIR = str(DATA_ROOT / "test_patches_13")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_13_ens_by_logratio_New")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 13 # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = True
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # stable sigmoid
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # log r(x) = logit(x) + log(pi0/pi1)
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # bounded suitability in (0,1), not interpreted as prevalence probability
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    # IMPORTANT: evaluate metrics on the SAME bounded score everywhere
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# - metrics computed on bounded score sigmoid(log_ratio)
# - fold-specific priors stored in checkpoint
# - returns OOF bounded scores for that fold's val split
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score) using fold checkpoints
# - computes per-model log_ratio using stored fold priors
# - ensembles in log space (default) for stability
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                # weighted mean of log-ratios (geometric mean of ratios)
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                # ratio-space average (less stable)
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights (OK for ensembling) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


# Patch size: 33


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_33")
TEST_DIR = str(DATA_ROOT / "test_patches_33")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_33_ens_by_logratio_New")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 33  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # stable sigmoid
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # log r(x) = logit(x) + log(pi0/pi1)
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # bounded suitability in (0,1), not interpreted as prevalence probability
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    # IMPORTANT: evaluate metrics on the SAME bounded score everywhere
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# - metrics computed on bounded score sigmoid(log_ratio)
# - fold-specific priors stored in checkpoint
# - returns OOF bounded scores for that fold's val split
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score) using fold checkpoints
# - computes per-model log_ratio using stored fold priors
# - ensembles in log space (default) for stability
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                # weighted mean of log-ratios (geometric mean of ratios)
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                # ratio-space average (less stable)
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights (OK for ensembling) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_33_ens_by_logratio_New")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_33_logspace_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 33
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (unbounded, but clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain fold-specific priors (pi0/pi1)
# Training code (updated) no longer needs:
#  - class_priors.npy
#  - suitability_offset_c.npy
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1)
      - fold_ids.npy
      - fold_weights_loss.npy

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logpi_t: (K,) where logpi_t[k] = log(pi0_k/pi1_k) from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # harmless at eval (model.eval disables dropout)
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)

    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (UPDATED: fold-specific priors are in each checkpoint)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection (UPDATED)
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )


# Patch size: 65


In [ ]:
import os
from typing import List, Dict, Tuple

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.metrics import roc_auc_score

# =========================================================
# CONFIG
# =========================================================

CV_DIR = str(DATA_ROOT / "cv_patches_65")
TEST_DIR = str(DATA_ROOT / "test_patches_65")

CV_X_PATH = os.path.join(CV_DIR, "X.npy")
CV_M_PATH = os.path.join(CV_DIR, "M.npy")
CV_Y_PATH = os.path.join(CV_DIR, "y.npy")
CV_FOLD_PATH = os.path.join(CV_DIR, "fold.npy")

TEST_X_PATH = os.path.join(TEST_DIR, "X.npy")
TEST_M_PATH = os.path.join(TEST_DIR, "M.npy")
TEST_Y_PATH = os.path.join(TEST_DIR, "y.npy")

MODEL_DIR = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_65_ens_by_logratio_New")

BATCH_SIZE = 256
NUM_WORKERS = 0

MAX_EPOCHS = 100
PATIENCE = 10

LR = 1e-3
WEIGHT_DECAY = 1e-3

CLIP_LOGITS = 10.0
SEED = 42

EMB_DIM = 32
HIDDEN_DIMS = [32]
DROPOUT = 0.35

PATCH_SIZE = 65  # 13x13 patches

# Data augmentation
USE_FLIPS = True
NOISE_STD = 0.08

# Selection rule: loss window + best Boyce (computed on bounded score)
USE_AUC_FLOOR = False
AUC_FLOOR = 0.80
LOSS_WINDOW_DELTA = 0.03

# Numeric stability
# Clamp log-ratio before applying sigmoid for bounded score
MAX_ABS_LOG_RATIO = 50.0

# Ensemble choice: weighted average of log-ratios (more stable than averaging ratios)
ENSEMBLE_IN_LOGSPACE = True


# =========================================================
# Device
# =========================================================
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================================================
# Metrics: Boyce + AUC (expects any real-valued score)
# =========================================================
def _average_ranks(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty_like(order, dtype=float)
    ranks[order] = np.arange(1, len(x) + 1, dtype=float)
    sx = x[order]
    i = 0
    while i < len(x):
        j = i + 1
        while j < len(x) and sx[j] == sx[i]:
            j += 1
        if j - i > 1:
            avg = (i + 1 + j) / 2.0
            ranks[order[i:j]] = avg
        i = j
    return ranks


def _spearman_tieaware(x, y) -> float:
    x = np.asarray(x, float)
    y = np.asarray(y, float)
    if x.size < 3 or y.size < 3:
        return np.nan
    rx, ry = _average_ranks(x), _average_ranks(y)
    if np.all(rx == rx[0]) or np.all(ry == ry[0]):
        return np.nan
    return float(np.corrcoef(rx, ry)[0, 1])


def continuous_boyce(y_true, scores, nbins_max=20, min_per_group=10):
    y = np.asarray(y_true).astype(int)
    s = np.asarray(scores, dtype=float)

    s_bg = s[y == 0]
    s_pr = s[y == 1]
    if (s_bg.size < min_per_group) or (s_pr.size < min_per_group):
        return np.nan

    uq = np.unique(s_bg[np.isfinite(s_bg)])
    if uq.size < 3:
        return np.nan
    nb = min(nbins_max, max(3, uq.size - 1))

    qs = np.quantile(s_bg, np.linspace(0.0, 1.0, nb + 1))
    qs[0] -= 1e-12
    qs[-1] += 1e-12

    pratio, centers = [], []
    Lb, Lp = float(len(s_bg)), float(len(s_pr))
    for a, b in zip(qs[:-1], qs[1:]):
        in_bg = (s_bg >= a) & (s_bg < b)
        nbk = int(in_bg.sum())
        if nbk == 0:
            continue
        in_pr = (s_pr >= a) & (s_pr < b)
        npk = int(in_pr.sum())
        pratio.append((npk / Lp) / (nbk / Lb))
        centers.append(0.5 * (a + b))

    if len(pratio) < 3:
        return np.nan
    return _spearman_tieaware(np.asarray(centers), np.asarray(pratio))


def compute_auc(y_true, scores):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size < 2 or np.unique(y).size < 2:
        return np.nan
    return float(roc_auc_score(y, s))


def compute_boyce_and_auc(y_true, scores, nbins_boyce=20):
    y = np.asarray(y_true, int)
    s = np.asarray(scores, float)
    m = np.isfinite(s)
    y, s = y[m], s[m]
    if y.size == 0:
        return dict(Boyce=np.nan, ROC_AUC=np.nan)
    return dict(
        Boyce=continuous_boyce(y, s, nbins_max=nbins_boyce),
        ROC_AUC=compute_auc(y, s),
    )


# =========================================================
# Helpers: stable bounded score from logits via log-ratio
# =========================================================
def sigmoid_np(x: np.ndarray) -> np.ndarray:
    # stable sigmoid
    x = np.asarray(x, dtype=np.float64)
    x = np.clip(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return 1.0 / (1.0 + np.exp(-x))


def logits_to_logratio(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # log r(x) = logit(x) + log(pi0/pi1)
    eps = 1e-12
    pi0 = float(np.clip(pi0, eps, 1.0 - eps))
    pi1 = float(np.clip(pi1, eps, 1.0 - eps))
    return logits_np + np.log(pi0 / pi1)


def logits_to_bounded_score(logits_np: np.ndarray, pi0: float, pi1: float) -> np.ndarray:
    # bounded suitability in (0,1), not interpreted as prevalence probability
    log_ratio = logits_to_logratio(logits_np, pi0, pi1)
    return sigmoid_np(log_ratio)


def compute_epoch_metrics_from_logits(logits_np, y_np, loss_val, pi0, pi1):
    # IMPORTANT: evaluate metrics on the SAME bounded score everywhere
    score01 = logits_to_bounded_score(logits_np, pi0, pi1)
    mets = compute_boyce_and_auc(y_np, score01)
    mets["loss"] = float(loss_val)
    return mets


def set_seed(seed=42):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def make_cv_indices(fold_cv: np.ndarray) -> List[int]:
    vals = np.unique(fold_cv)
    return sorted(int(v) for v in vals)


# =========================================================
# Dataset + Models
# =========================================================
def augment_patch(values: torch.Tensor, mask: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    if USE_FLIPS:
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[1])
            mask = torch.flip(mask, dims=[1])
        if torch.rand(1).item() < 0.5:
            values = torch.flip(values, dims=[2])
            mask = torch.flip(mask, dims=[2])

    if NOISE_STD > 0.0:
        noise = torch.randn_like(values) * NOISE_STD
        values = values + noise

    return values, mask


class PatchDREDataset(Dataset):
    def __init__(self, X_values, X_masks, y, indices, train: bool):
        super().__init__()
        self.Xv = X_values.astype(np.float32)
        self.Xm = X_masks.astype(np.float32)
        self.y = y.astype(np.float32)
        self.indices = np.asarray(indices, dtype=np.int64)
        self.train = train

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i: int):
        idx = self.indices[i]
        xv = torch.from_numpy(self.Xv[idx])
        xm = torch.from_numpy(self.Xm[idx])
        y = torch.tensor(self.y[idx], dtype=torch.float32)
        if self.train:
            xv, xm = augment_patch(xv, xm)
        return xv, xm, y


class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(256, 128, 64), dropout=0.2, clip_logits=None):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        z = self.proj(h)
        return z


class CNNDRE(nn.Module):
    def __init__(self, in_value_channels, emb_dim=64, hidden_dims=(128, 64), dropout=0.2, clip_logits=10.0):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(in_dim=emb_dim, hidden_dims=hidden_dims, dropout=dropout, clip_logits=clip_logits)

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        logits = self.dre_head(z)
        return logits


# =========================================================
# Training per fold
# - metrics computed on bounded score sigmoid(log_ratio)
# - fold-specific priors stored in checkpoint
# - returns OOF bounded scores for that fold's val split
# =========================================================
def train_and_save_fold_model_cnn(
    fold_id: int,
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y_cv: np.ndarray,
    fold_cv: np.ndarray,
    model_dir: str,
) -> Tuple[Dict[str, float], np.ndarray, np.ndarray]:
    os.makedirs(model_dir, exist_ok=True)

    f = fold_cv.astype(int)
    train_idx = np.where(f != fold_id)[0]
    val_idx = np.where(f == fold_id)[0]

    # fold-specific priors from TRAIN ONLY
    pi1_fold = float((y_cv[train_idx] == 1).mean())
    pi0_fold = 1.0 - pi1_fold
    print(f"\n[fold {fold_id}] train N={len(train_idx)}, val N={len(val_idx)} | priors pi1={pi1_fold:.6f}, pi0={pi0_fold:.6f}")

    ds_tr = PatchDREDataset(X_values, X_masks, y_cv, train_idx, train=True)
    ds_val = PatchDREDataset(X_values, X_masks, y_cv, val_idx, train=False)

    dl_tr = DataLoader(ds_tr, batch_size=BATCH_SIZE, shuffle=True,
                       num_workers=NUM_WORKERS, drop_last=False)
    dl_val = DataLoader(ds_val, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, drop_last=False)

    in_value_channels = X_values.shape[1]
    model = CNNDRE(
        in_value_channels=in_value_channels,
        emb_dim=EMB_DIM,
        hidden_dims=HIDDEN_DIMS,
        dropout=DROPOUT,
        clip_logits=CLIP_LOGITS,
    ).to(device)

    bce = nn.BCEWithLogitsLoss()
    opt = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, opt_obj=None):
        train = opt_obj is not None
        model.train() if train else model.eval()

        total_loss = 0.0
        all_logits, all_y = [], []

        for xv, xm, yb in loader:
            xv, xm, yb = xv.to(device), xm.to(device), yb.to(device)
            if train:
                opt_obj.zero_grad(set_to_none=True)

            with torch.set_grad_enabled(train):
                logits = model(xv, xm)
                loss = bce(logits, yb)
                if train:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt_obj.step()

            total_loss += loss.item() * yb.size(0)
            all_logits.append(logits.detach().cpu().numpy())
            all_y.append(yb.detach().cpu().numpy())

        total_loss /= len(loader.dataset)
        all_logits = np.concatenate(all_logits)
        all_y = np.concatenate(all_y)

        mets = compute_epoch_metrics_from_logits(all_logits, all_y, total_loss, pi0_fold, pi1_fold)
        return mets, all_logits, all_y

    # Selection: among epochs passing AUC floor, pick max Boyce in (loss <= loss_min + delta)
    candidates = []
    best_loss_seen = np.inf
    no_improve = 0

    for ep in range(1, MAX_EPOCHS + 1):
        tr, _, _ = run_epoch(dl_tr, opt)
        va, val_logits, val_y = run_epoch(dl_val, None)

        print(
            f"[fold {fold_id}] epoch {ep:03d} | "
            f"train loss {tr['loss']:.4f}, AUC {tr['ROC_AUC']:.4f}, Boyce {tr['Boyce']:.4f} | "
            f"val loss {va['loss']:.4f}, AUC {va['ROC_AUC']:.4f}, Boyce {va['Boyce']:.4f}"
        )

        auc_ok = True
        if USE_AUC_FLOOR:
            auc_ok = np.isfinite(va["ROC_AUC"]) and (va["ROC_AUC"] >= AUC_FLOOR)

        if auc_ok:
            state = {
                "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
                "in_value_channels": in_value_channels,
                "patch_size": PATCH_SIZE,
                "emb_dim": EMB_DIM,
                "hidden_dims": HIDDEN_DIMS,
                "pi0": float(pi0_fold),
                "pi1": float(pi1_fold),
            }
            candidates.append(
                {
                    "loss": float(va["loss"]),
                    "boyce": float(va["Boyce"]) if np.isfinite(va["Boyce"]) else np.nan,
                    "auc": float(va["ROC_AUC"]) if np.isfinite(va["ROC_AUC"]) else np.nan,
                    "state": state,
                    "val_logits": val_logits,
                    "val_y": val_y,
                    "epoch": ep,
                }
            )

            if va["loss"] < best_loss_seen - 1e-9:
                best_loss_seen = float(va["loss"])
                no_improve = 0
            else:
                no_improve += 1
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"[fold {fold_id}] early stopping at epoch {ep}")
            break

    if len(candidates) == 0:
        raise RuntimeError(
            f"[fold {fold_id}] No epoch met AUC floor (AUC_FLOOR={AUC_FLOOR}). "
            f"Lower AUC_FLOOR or set USE_AUC_FLOOR=False."
        )

    loss_min = min(c["loss"] for c in candidates)
    thresh = loss_min + LOSS_WINDOW_DELTA
    window = [c for c in candidates if c["loss"] <= thresh]

    def key(c):
        b = c["boyce"]
        b_val = -np.inf if (b is None or not np.isfinite(b)) else float(b)
        return (b_val, -c["loss"])

    best = max(window, key=key)

    ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fold_id}.pt")
    torch.save(best["state"], ckpt_path)
    print(
        f"[fold {fold_id}] selected epoch={best['epoch']} | "
        f"loss={best['loss']:.4f}, Boyce={best['boyce']:.4f}, AUC={best['auc']:.4f} | "
        f"saved -> {ckpt_path}"
    )

    # Return OOF bounded scores for this fold's val subset
    val_logits = best["val_logits"]
    val_y = best["val_y"]
    val_score01 = logits_to_bounded_score(val_logits, best["state"]["pi0"], best["state"]["pi1"])
    best_val = {"Boyce": best["boyce"], "ROC_AUC": best["auc"], "loss": best["loss"]}
    return best_val, val_score01, val_idx


# =========================================================
# Ensemble prediction (bounded score) using fold checkpoints
# - computes per-model log_ratio using stored fold priors
# - ensembles in log space (default) for stability
# =========================================================
def ensemble_predict_on_indices_cnn(
    X_values: np.ndarray,
    X_masks: np.ndarray,
    y: np.ndarray,
    indices: np.ndarray,
    model_dir: str,
    fold_ids: List[int],
    weights: np.ndarray,
):
    in_value_channels = X_values.shape[1]

    models = []
    per_model_logpi = []
    for fid in fold_ids:
        ckpt_path = os.path.join(model_dir, f"cnn_dre_fold_{fid}.pt")
        state = torch.load(ckpt_path, map_location="cpu")

        if state["in_value_channels"] != in_value_channels:
            raise RuntimeError(f"Value channel mismatch for fold {fid}")
        if state.get("patch_size", PATCH_SIZE) != PATCH_SIZE:
            raise RuntimeError(f"Patch size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Missing fold priors in checkpoint for fold {fid} (pi0/pi1).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=state.get("emb_dim", EMB_DIM),
            hidden_dims=tuple(state.get("hidden_dims", HIDDEN_DIMS)),
            dropout=DROPOUT,
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        per_model_logpi.append(np.log(pi0_k / pi1_k))

    # normalize weights
    w = np.asarray(weights, dtype=np.float64)
    if (not np.isfinite(w).all()) or w.sum() <= 0:
        w = np.ones(len(fold_ids), dtype=np.float64) / len(fold_ids)
    else:
        w = w / w.sum()

    weights_t = torch.tensor(w, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(np.asarray(per_model_logpi, dtype=np.float32), dtype=torch.float32, device=device)

    idx = np.asarray(indices, dtype=np.int64)

    all_score01, all_logratio, all_y = [], [], []

    with torch.no_grad():
        for start in range(0, len(idx), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(idx))
            idx_batch = idx[start:end]

            xv = torch.from_numpy(X_values[idx_batch].astype(np.float32)).to(device)
            xm = torch.from_numpy(X_masks[idx_batch].astype(np.float32)).to(device)
            yb = torch.from_numpy(y[idx_batch].astype(np.float32)).to(device)

            # collect per-model log-ratios
            logratios = []
            for k, model in enumerate(models):
                logits_k = model(xv, xm)
                log_ratio_k = logits_k + logpi_t[k]
                log_ratio_k = torch.clamp(log_ratio_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                logratios.append(log_ratio_k)

            # ensemble
            if ENSEMBLE_IN_LOGSPACE:
                # weighted mean of log-ratios (geometric mean of ratios)
                log_ratio_ens = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    log_ratio_ens += weights_t[k] * logratios[k]
            else:
                # ratio-space average (less stable)
                ratio_sum = torch.zeros(xv.size(0), device=device)
                for k in range(len(models)):
                    ratio_sum += weights_t[k] * torch.exp(logratios[k])
                log_ratio_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

            log_ratio_ens = torch.clamp(log_ratio_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
            score01 = torch.sigmoid(log_ratio_ens)

            all_score01.append(score01.cpu().numpy())
            all_logratio.append(log_ratio_ens.cpu().numpy())
            all_y.append(yb.cpu().numpy())

    return (
        np.concatenate(all_score01),
        np.concatenate(all_logratio),
        np.concatenate(all_y),
    )


# =========================================================
# MAIN
# =========================================================
if __name__ == "__main__":
    set_seed(SEED)
    os.makedirs(MODEL_DIR, exist_ok=True)

    # ----------- load CV data -----------
    X_cv = np.load(CV_X_PATH)
    M_cv = np.load(CV_M_PATH)
    y_cv = np.load(CV_Y_PATH).astype(np.float32)
    fold_cv_raw = np.load(CV_FOLD_PATH)

    print("[CV] X shape:", X_cv.shape)
    print("[CV] M shape:", M_cv.shape)
    print("[CV] y shape:", y_cv.shape)
    print("[CV] fold shape:", fold_cv_raw.shape)

    if X_cv.shape[0] != y_cv.shape[0] or M_cv.shape[0] != y_cv.shape[0]:
        raise ValueError("Mismatch between CV X/M and y lengths")

    if not np.isfinite(fold_cv_raw).all():
        raise ValueError("fold.npy contains NaN/inf; CV split must be defined for all rows")
    fold_cv = fold_cv_raw.astype(int)

    bad = ~np.isfinite(X_cv)
    if bad.any():
        print(f"[CV] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_cv[bad] = 0.0
    bad_m = ~np.isfinite(M_cv)
    if bad_m.any():
        print(f"[CV] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_cv[bad_m] = 0.0

    # ----------- load TEST data -----------
    X_test = np.load(TEST_X_PATH)
    M_test = np.load(TEST_M_PATH)
    y_test = np.load(TEST_Y_PATH).astype(np.float32)

    print("[TEST] X shape:", X_test.shape)
    print("[TEST] M shape:", M_test.shape)
    print("[TEST] y shape:", y_test.shape)

    if X_test.shape[0] != y_test.shape[0] or M_test.shape[0] != y_test.shape[0]:
        raise ValueError("Mismatch between TEST X/M and y lengths")

    bad = ~np.isfinite(X_test)
    if bad.any():
        print(f"[TEST] Warning: {int(bad.sum())} non-finite X values set to 0.")
        X_test[bad] = 0.0
    bad_m = ~np.isfinite(M_test)
    if bad_m.any():
        print(f"[TEST] Warning: {int(bad_m.sum())} non-finite M values set to 0.")
        M_test[bad_m] = 0.0

    # ----------- CV training over folds -----------
    fold_ids = make_cv_indices(fold_cv)
    print("Folds:", fold_ids)

    # OOF bounded scores (0..1) for *metrics comparability*
    oof_score01 = np.full_like(y_cv, np.nan, dtype=float)

    fold_aucs = []
    fold_losses = []

    for fid in fold_ids:
        best_val, val_score01, val_idx = train_and_save_fold_model_cnn(
            fid, X_cv, M_cv, y_cv, fold_cv, MODEL_DIR
        )
        fold_aucs.append(best_val["ROC_AUC"])
        fold_losses.append(best_val["loss"])
        oof_score01[val_idx] = val_score01

    # ----------- CV metrics (OOF bounded score) -----------
    cv_metrics = compute_boyce_and_auc(y_cv, oof_score01)
    print("\n[CV] OOF metrics (bounded score):", cv_metrics)
    print("[CV] per-fold best AUCs:", fold_aucs)
    print("[CV] per-fold chosen losses:", fold_losses)

    # ----------- Loss-based weights (OK for ensembling) -----------
    loss_arr = np.asarray(fold_losses, dtype=float)
    if (not np.isfinite(loss_arr).all()) or loss_arr.size == 0:
        weights = np.ones(len(fold_ids), dtype=float) / len(fold_ids)
        print("\n[Ensemble] Losses invalid; using equal weights.")
    else:
        loss_min = float(np.min(loss_arr))
        w = np.exp(-(loss_arr - loss_min))
        w = w / w.sum()
        weights = w
        print("\n[Ensemble] Loss-based weights exp(-(loss-loss_min)):", weights)

    np.save(os.path.join(MODEL_DIR, "fold_ids.npy"), np.array(fold_ids, dtype=int))
    np.save(os.path.join(MODEL_DIR, "fold_losses.npy"), loss_arr)
    np.save(os.path.join(MODEL_DIR, "fold_weights_loss.npy"), weights)
    np.save(os.path.join(MODEL_DIR, "oof_score01.npy"), oof_score01)
    print("[Saved] fold_ids.npy, fold_losses.npy, fold_weights_loss.npy, oof_score01.npy")

    # ----------- External test ensemble (bounded score) -----------
    test_idx = np.arange(len(y_test))
    print("External test N =", len(test_idx))

    test_score01, test_logratio, test_y = ensemble_predict_on_indices_cnn(
        X_values=X_test,
        X_masks=M_test,
        y=y_test,
        indices=test_idx,
        model_dir=MODEL_DIR,
        fold_ids=fold_ids,
        weights=weights,
    )

    test_metrics = compute_boyce_and_auc(test_y, test_score01)
    print("\n[Test] Ensemble metrics (bounded score):", test_metrics)

    np.save(os.path.join(MODEL_DIR, "test_score01.npy"), test_score01)
    np.save(os.path.join(MODEL_DIR, "test_logratio_ens.npy"), test_logratio)
    print("[Test] Saved test_score01.npy, test_logratio_ens.npy")


In [ ]:
import os
import math
from pathlib import Path
from typing import List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from tqdm import tqdm

# -------------------------------------------------------------
# CONFIG — MUST MATCH TRAINING (UPDATED)
# -------------------------------------------------------------
PROCESSED_RASTER_DIR = str(RASTER_DIR)
MODEL_DIR            = str(OUTPUT_ROOT / "classifier" / "cnn_dre_patch_classifier_65_ens_by_logratio_New")
LANDMASK_PATH        = str(LANDMASK_PATH)

OUT_TIF = str(OUTPUT_ROOT / "classifier" / "dengue_cnn_suitability_65_logspace_ens.tif")

BATCH_SIZE     = 512
TILE_SIZE      = 128
PATCH_SIZE     = 65
RASTER_PATTERN = "*.tif"

DST_NODATA = -9999.0

# Must match training
MAX_ABS_LOG_RATIO = 50.0
CLIP_LOGITS = 10.0
DROPOUT = 0.35

# Output control
# - If True: write log-ratio map (unbounded, but clamped to +/- MAX_ABS_LOG_RATIO)
# - If False: write bounded score in (0,1): sigmoid(log_ratio)
WRITE_LOGR = False

# Ensemble mode should match your training-time inference choice
ENSEMBLE_IN_LOGSPACE = True  # weighted mean of log-ratios (recommended)

# -------------------------------------------------------------
# Device
# -------------------------------------------------------------
if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Prediction device:", device)


# -------------------------------------------------------------
# Model definitions — MUST MATCH TRAINING
# -------------------------------------------------------------
class MLPDRE(nn.Module):
    def __init__(self, in_dim: int, hidden_dims=(32,), dropout=0.35, clip_logits: Optional[float] = 10.0):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            if dropout and dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self.clip_logits = clip_logits

    def forward(self, x):
        logits = self.net(x).squeeze(-1)
        if self.clip_logits is not None:
            logits = torch.clamp(logits, -self.clip_logits, self.clip_logits)
        return logits


class PatchEncoder13(nn.Module):
    def __init__(self, in_value_channels: int, emb_dim: int = 32):
        super().__init__()
        in_channels = 2 * in_value_channels
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.Conv2d(16, 16, 3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True),

            nn.MaxPool2d(2, 2),  # 13→6

            nn.Conv2d(16, 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.AdaptiveAvgPool2d(1),
        )
        self.proj = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32, emb_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x_val, x_mask):
        x_val = x_val * x_mask
        x_in = torch.cat([x_val, x_mask], dim=1)
        h = self.features(x_in)
        return self.proj(h)


class CNNDRE(nn.Module):
    def __init__(
        self,
        in_value_channels: int,
        emb_dim: int = 32,
        hidden_dims=(32,),
        dropout: float = 0.35,
        clip_logits: Optional[float] = 10.0,
    ):
        super().__init__()
        self.encoder = PatchEncoder13(in_value_channels, emb_dim)
        self.dre_head = MLPDRE(
            in_dim=emb_dim,
            hidden_dims=hidden_dims,
            dropout=dropout,
            clip_logits=clip_logits,
        )

    def forward(self, x_val, x_mask):
        z = self.encoder(x_val, x_mask)
        return self.dre_head(z)


# -------------------------------------------------------------
# Stable sigmoid (bounded score from log-ratio)
# -------------------------------------------------------------
def sigmoid_torch(x: torch.Tensor) -> torch.Tensor:
    x = torch.clamp(x, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
    return torch.sigmoid(x)


# -------------------------------------------------------------
# Load ensemble: fold checkpoints contain fold-specific priors (pi0/pi1)
# Training code (updated) no longer needs:
#  - class_priors.npy
#  - suitability_offset_c.npy
# -------------------------------------------------------------
def load_cnn_ensemble_models(model_dir: str) -> Tuple[List[CNNDRE], torch.Tensor, torch.Tensor, int, int]:
    """
    Expects in model_dir:
      - cnn_dre_fold_<fid>.pt  (each contains pi0, pi1)
      - fold_ids.npy
      - fold_weights_loss.npy

    Returns:
      models: List[CNNDRE]
      weights_t: (K,)
      logpi_t: (K,) where logpi_t[k] = log(pi0_k/pi1_k) from that fold checkpoint
      in_value_channels, patch_size
    """
    mdir = Path(model_dir)

    fold_ids_path = mdir / "fold_ids.npy"
    if not fold_ids_path.exists():
        raise FileNotFoundError(f"Missing {fold_ids_path}")
    fold_ids = np.load(fold_ids_path).astype(int)

    w_path = mdir / "fold_weights_loss.npy"
    if not w_path.exists():
        raise FileNotFoundError(f"Missing {w_path}")
    weights = np.load(w_path).astype(np.float32)

    if fold_ids.size != weights.size:
        raise ValueError(f"fold_ids size {fold_ids.size} != weights size {weights.size}")

    if not np.isfinite(weights).any() or float(np.nansum(weights)) <= 0:
        weights = np.ones_like(weights) / len(weights)
        print("[Ensemble] Invalid weights -> using equal weights.")
    else:
        w = np.maximum(weights, 1e-8)
        weights = w / w.sum()

    print("Loaded folds:", fold_ids.tolist())
    print("Loaded loss-weights:", weights.tolist())

    # Read an example ckpt for architecture params
    example_ckpt = torch.load(mdir / f"cnn_dre_fold_{fold_ids[0]}.pt", map_location="cpu")
    in_value_channels = int(example_ckpt["in_value_channels"])
    patch_size = int(example_ckpt.get("patch_size", PATCH_SIZE))
    emb_dim = int(example_ckpt.get("emb_dim", 32))
    hidden_dims = tuple(example_ckpt.get("hidden_dims", [32]))

    models: List[CNNDRE] = []
    logpi_list: List[float] = []

    for fid in fold_ids:
        ckpt_path = mdir / f"cnn_dre_fold_{fid}.pt"
        state = torch.load(ckpt_path, map_location="cpu")

        if int(state["in_value_channels"]) != in_value_channels:
            raise RuntimeError(f"in_value_channels mismatch for fold {fid}")
        if int(state.get("patch_size", PATCH_SIZE)) != patch_size:
            raise RuntimeError(f"patch_size mismatch for fold {fid}")
        if ("pi0" not in state) or ("pi1" not in state):
            raise RuntimeError(f"Checkpoint for fold {fid} missing pi0/pi1 (fold-specific priors).")

        model = CNNDRE(
            in_value_channels=in_value_channels,
            emb_dim=int(state.get("emb_dim", emb_dim)),
            hidden_dims=tuple(state.get("hidden_dims", hidden_dims)),
            dropout=DROPOUT,       # harmless at eval (model.eval disables dropout)
            clip_logits=CLIP_LOGITS,
        ).to(device)
        model.load_state_dict(state["model_state"])
        model.eval()
        models.append(model)

        pi0_k = float(state["pi0"])
        pi1_k = float(state["pi1"])
        logpi_list.append(float(np.log(pi0_k / pi1_k)))

    weights_t = torch.tensor(weights, dtype=torch.float32, device=device)
    logpi_t = torch.tensor(logpi_list, dtype=torch.float32, device=device)

    return models, weights_t, logpi_t, in_value_channels, patch_size


# -------------------------------------------------------------
# Raster medians for imputation
# -------------------------------------------------------------
def compute_raster_medians(rasters: List[rasterio.DatasetReader], downscale: int = 8) -> np.ndarray:
    meds = []
    for r in rasters:
        out_h = max(1, r.height // downscale)
        out_w = max(1, r.width // downscale)
        thumb = r.read(
            1,
            out_shape=(1, out_h, out_w),
            resampling=Resampling.nearest,
        ).astype("float32")

        nodata = r.nodata
        if nodata is not None:
            thumb = np.where(thumb == nodata, np.nan, thumb)

        meds.append(float(np.nanmedian(thumb)))
    return np.array(meds, dtype=np.float32)


# -------------------------------------------------------------
# Extract patches for a tile's land pixels
# -------------------------------------------------------------
def extract_patches_for_tile(
    rasters: List[rasterio.DatasetReader],
    raster_medians: np.ndarray,
    row0: int,
    row1: int,
    col0: int,
    col1: int,
    rr_rel: np.ndarray,
    cc_rel: np.ndarray,
    patch_size: int,
) -> Tuple[np.ndarray, np.ndarray]:
    half = patch_size // 2
    template = rasters[0]
    H, W = template.height, template.width
    C = len(rasters)

    N_land = rr_rel.shape[0]
    values = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)
    masks  = np.zeros((N_land, C, patch_size, patch_size), dtype=np.float32)

    r0_big = max(0, row0 - half)
    r1_big = min(H, row1 + half)
    c0_big = max(0, col0 - half)
    c1_big = min(W, col1 + half)
    h_big = r1_big - r0_big
    w_big = c1_big - c0_big
    big_win = Window(col_off=c0_big, row_off=r0_big, width=w_big, height=h_big)

    for c_idx, src in enumerate(rasters):
        med_val = raster_medians[c_idx]
        arr_big = src.read(1, window=big_win).astype("float32")

        nodata = src.nodata
        if nodata is not None:
            arr_big = np.where(arr_big == nodata, np.nan, arr_big)

        for k in range(N_land):
            r_center = row0 + rr_rel[k]
            c_center = col0 + cc_rel[k]

            r_center_big = r_center - r0_big
            c_center_big = c_center - c0_big

            rs = r_center_big - half
            re = r_center_big + half + 1
            cs = c_center_big - half
            ce = c_center_big + half + 1

            rs_c = max(0, rs)
            cs_c = max(0, cs)
            re_c = min(h_big, re)
            ce_c = min(w_big, ce)

            if (re_c - rs_c) <= 0 or (ce_c - cs_c) <= 0:
                patch = np.full((patch_size, patch_size), np.nan, dtype=np.float32)
            else:
                patch = arr_big[rs_c:re_c, cs_c:ce_c]
                pad_top = rs_c - rs
                pad_left = cs_c - cs
                pad_bottom = patch_size - pad_top - patch.shape[0]
                pad_right = patch_size - pad_left - patch.shape[1]
                patch = np.pad(
                    patch,
                    pad_width=((pad_top, pad_bottom), (pad_left, pad_right)),
                    mode="constant",
                    constant_values=np.nan,
                ).astype("float32")

            m = np.isfinite(patch)
            filled = np.where(m, patch, med_val).astype("float32")

            values[k, c_idx] = filled
            masks[k, c_idx] = m.astype("float32")

    return values, masks


# -------------------------------------------------------------
# Main prediction over rasters
# -------------------------------------------------------------
def predict_suitability_map_cnn(
    raster_dir: str,
    model_dir: str,
    landmask_path: str,
    out_tif: str,
    pattern: str = "*.tif",
    tile_size: int = 256,
    batch_size: int = 512,
):
    raster_dir = Path(raster_dir)
    cov_paths = sorted(raster_dir.glob(pattern))
    if not cov_paths:
        raise FileNotFoundError(f"No rasters in {raster_dir} matching pattern '{pattern}'")

    rasters = [rasterio.open(p) for p in cov_paths]
    n_bands = len(rasters)
    print(f"Using {n_bands} covariate rasters:")
    for p in cov_paths:
        print("  -", p.name)

    template = rasters[0]
    height, width = template.height, template.width
    transform = template.transform
    profile = template.profile.copy()

    for src in rasters[1:]:
        if src.width != width or src.height != height:
            raise ValueError(f"Raster size mismatch between {cov_paths[0].name} and {src.name}")
        if src.transform != transform:
            raise ValueError(f"Transform mismatch between {cov_paths[0].name} and {src.name}")

    # Landmask
    with rasterio.open(landmask_path) as lm:
        if lm.width != width or lm.height != height:
            raise ValueError("Landmask and covariates have different sizes.")
        if lm.transform != transform:
            raise ValueError("Landmask and covariates have different geotransforms.")
        landmask = lm.read(1)

    # Load ensemble (UPDATED: fold-specific priors are in each checkpoint)
    models, weights_t, logpi_t, in_value_channels, patch_size = load_cnn_ensemble_models(model_dir=model_dir)
    if in_value_channels != n_bands:
        raise RuntimeError(
            f"Number of covariate rasters ({n_bands}) != in_value_channels ({in_value_channels}). "
            f"Check raster order matches training."
        )
    if patch_size != PATCH_SIZE:
        raise RuntimeError(
            f"Patch size from checkpoint ({patch_size}) != PATCH_SIZE in config ({PATCH_SIZE})."
        )

    # Precompute medians
    raster_medians = compute_raster_medians(rasters, downscale=8)
    print("Raster medians:", raster_medians)

    # Store log-ratio, then optionally convert to bounded score
    log_r_map = np.full((height, width), np.nan, dtype=np.float32)

    n_rows_tiles = math.ceil(height / tile_size)
    n_cols_tiles = math.ceil(width / tile_size)
    total_tiles = n_rows_tiles * n_cols_tiles

    with tqdm(total=total_tiles, desc="Tiles", unit="tile") as pbar:
        for row0 in range(0, height, tile_size):
            row1 = min(row0 + tile_size, height)
            tile_h = row1 - row0

            for col0 in range(0, width, tile_size):
                col1 = min(col0 + tile_size, width)
                tile_w = col1 - col0

                lm_tile = landmask[row0:row1, col0:col1]
                is_land = (lm_tile == 1)
                if not np.any(is_land):
                    pbar.update(1)
                    continue

                rr_rel, cc_rel = np.where(is_land)
                N_land = rr_rel.shape[0]

                vals_tile, masks_tile = extract_patches_for_tile(
                    rasters=rasters,
                    raster_medians=raster_medians,
                    row0=row0,
                    row1=row1,
                    col0=col0,
                    col1=col1,
                    rr_rel=rr_rel,
                    cc_rel=cc_rel,
                    patch_size=patch_size,
                )

                tile_logr = np.zeros(N_land, dtype=np.float32)

                with torch.no_grad():
                    for start in range(0, N_land, batch_size):
                        end = min(start + batch_size, N_land)
                        xv = torch.from_numpy(vals_tile[start:end]).to(device)
                        xm = torch.from_numpy(masks_tile[start:end]).to(device)

                        # Per-model log-ratio: log_r_k = logits_k + log(pi0_k/pi1_k)
                        logratios = []
                        for k, model in enumerate(models):
                            logits_k = model(xv, xm)  # (B,)
                            log_r_k = logits_k + logpi_t[k]
                            log_r_k = torch.clamp(log_r_k, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                            logratios.append(log_r_k)

                        # Ensemble
                        if ENSEMBLE_IN_LOGSPACE:
                            # weighted mean of log-ratios (geometric mean of ratios)
                            log_r_ens = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                log_r_ens += weights_t[k] * logratios[k]
                        else:
                            # ratio-space average (less stable)
                            ratio_sum = torch.zeros(xv.size(0), device=device, dtype=torch.float32)
                            for k in range(len(models)):
                                ratio_sum += weights_t[k] * torch.exp(logratios[k])
                            log_r_ens = torch.log(torch.clamp(ratio_sum, min=1e-30))

                        log_r_ens = torch.clamp(log_r_ens, -MAX_ABS_LOG_RATIO, MAX_ABS_LOG_RATIO)
                        tile_logr[start:end] = log_r_ens.detach().cpu().numpy().astype("float32")

                tile_logr_arr = np.full((tile_h, tile_w), np.nan, dtype=np.float32)
                tile_logr_arr[rr_rel, cc_rel] = tile_logr
                log_r_map[row0:row1, col0:col1] = tile_logr_arr

                pbar.update(1)

    # ---------------------------------------------------------
    # Output selection (UPDATED)
    # ---------------------------------------------------------
    if WRITE_LOGR:
        out_map = log_r_map.astype(np.float32)
        out_name = "log-ratio log_r (clamped)"
    else:
        # bounded suitability = sigmoid(log_r) in (0,1)
        x = torch.from_numpy(log_r_map.astype(np.float32))
        out_map = sigmoid_torch(x).cpu().numpy().astype(np.float32)
        out_name = "bounded suitability = sigmoid(log_r) in (0,1)"

    # Apply nodata
    out_map = np.where(np.isfinite(out_map), out_map, DST_NODATA).astype("float32")

    profile.update(dtype="float32", count=1, nodata=DST_NODATA, compress="lzw")
    out_tif = Path(out_tif)
    out_tif.parent.mkdir(parents=True, exist_ok=True)

    with rasterio.open(out_tif, "w", **profile) as dst:
        dst.write(out_map, 1)

    print(f"Saved {out_name} map to {str(out_tif)}")

    for r in rasters:
        r.close()


if __name__ == "__main__":
    predict_suitability_map_cnn(
        raster_dir=PROCESSED_RASTER_DIR,
        model_dir=MODEL_DIR,
        landmask_path=LANDMASK_PATH,
        out_tif=OUT_TIF,
        pattern=RASTER_PATTERN,
        tile_size=TILE_SIZE,
        batch_size=BATCH_SIZE,
    )
